# AI를 활용한 신약 개발 수업: v2026.09.15.

> by 보평고등학교 김태영

<img src="https://i.imgur.com/SbEKnJs.jpeg" width="800">

# <img src="https://i.imgur.com/EkC7gpR.png">

# Part 0: Target Protein 선정 및 항체 구조 확보


In [ ]:
# ChEMBL 클라이언트 라이브러리 설치
!pip install chembl_webresource_client
!pip install py3Dmol
!pip install biopython
!apt-get install -qq muscle  # MSA를 위한 MUSCLE
!apt-get install -qq hmmer    # HMM 기반 서열 검색

In [ ]:
#@title 🔍 ChEMBL 항체 검색 (NEW) { display-mode: "form" }
#@markdown **타겟 ChEMBL ID**를 입력하세요.

#@markdown > CHEMBL203 (EGFR), CHEMBL1824 (HER2), CHEMBL279 (VEGFR2)
target_chembl_id = "CHEMBL203"  #@param {type:"string"}

import requests, time, math
from requests.adapters import HTTPAdapter, Retry
import pandas as pd

# ---- 설정값 ----
# target_chembl_id 는 위 Form 입력값을 사용합니다.
DEBUG_LIMIT = None                       # 예: 10 으로 두고 먼저 파이프라인 검사
CHUNK_SIZE = 50                          # 한 번에 조회할 molecule id 개수
CONNECT_TIMEOUT, READ_TIMEOUT = 10, 30   # 타임아웃(초)
BASE = "https://www.ebi.ac.uk/chembl/api/data"
UA = "colab-chembl-antibody-search/1.0"

print("🔎 ChEMBL 항체 검색 시작...")
t0 = time.time()

# 세션 + 재시도
session = requests.Session()
retries = Retry(
    total=5, backoff_factor=0.6,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
session.mount("https://", HTTPAdapter(max_retries=retries))
session.headers.update({"User-Agent": UA})

def chembl_get_list(endpoint, params, limit_per=200):
    """ChEMBL REST endpoint를 안전하게 페이징 조회"""
    url = f"{BASE}/{endpoint}.json"
    out, offset = [], 0
    while True:
        p = dict(params)
        p.update({"limit": limit_per, "offset": offset})
        r = session.get(url, params=p, timeout=(CONNECT_TIMEOUT, READ_TIMEOUT))
        r.raise_for_status()
        data = r.json()
        # 엔드포인트마다 리스트 키가 다름
        for key in (endpoint + "s", "molecules", "mechanisms", "targets", "biotherapeutics"):
            if key in data:
                items = data[key]
                break
        else:
            items = []
        if not items:
            break
        out.extend(items)
        if len(items) < limit_per:
            break
        offset += limit_per
    return out

# 1) 타깃 정보
target_rows = chembl_get_list("target", {"target_chembl_id": target_chembl_id})
if not target_rows:
    raise RuntimeError(f"타겟 {target_chembl_id} 를 찾지 못했습니다.")
target_name = target_rows[0].get("pref_name") or target_rows[0].get("target_components", [{}])[0].get("component_description", "")
print(f"✅ 타겟: {target_name} (ChEMBL ID: {target_chembl_id})")

# 2) 타깃 → mechanism → 연관 분자 목록
mechs = chembl_get_list("mechanism", {"target_chembl_id": target_chembl_id})
mol_ids = sorted({m["molecule_chembl_id"] for m in mechs if m.get("molecule_chembl_id")})
if DEBUG_LIMIT:
    mol_ids = mol_ids[:DEBUG_LIMIT]
print(f"🔄 {len(mol_ids)}개 분자 분석 중... (배치 조회)")

# 3) molecule 메타데이터 배치 조회
def fetch_molecules(ids):
    res = []
    for i in range(0, len(ids), CHUNK_SIZE):
        chunk = ids[i:i+CHUNK_SIZE]
        mols = chembl_get_list("molecule", {"molecule_chembl_id__in": ",".join(chunk)})
        res.extend(mols)
        print(f"  • molecule {i+1:>3}-{min(i+CHUNK_SIZE, len(ids)):<3} / {len(ids)}")
    return res

molecules = fetch_molecules(mol_ids)

# 4) 항체 후보만 선별
def is_antibody(m):
    t = (m.get("molecule_type") or "").lower()
    name = (m.get("pref_name") or "").lower()
    # 분류값 또는 이름휴리스틱으로 항체 판단
    return (t == "antibody") or name.endswith("mab")

ab_mols = [m for m in molecules if is_antibody(m)]
ab_ids  = [m["molecule_chembl_id"] for m in ab_mols]
print(f"🧮 항체 후보 1차 필터: {len(ab_ids)}개")

# 5) (선택) biotherapeutic 상세를 가진 항체만 재확인
def fetch_biotherapeutic(ids):
    res = []
    for i in range(0, len(ids), CHUNK_SIZE):
        chunk = ids[i:i+CHUNK_SIZE]
        bios = chembl_get_list("biotherapeutic", {"molecule_chembl_id__in": ",".join(chunk)})
        res.extend(bios)
        print(f"  • biotherapeutic {i+1:>3}-{min(i+CHUNK_SIZE, len(ids)):<3} / {len(ids)}")
    return res

bios = fetch_biotherapeutic(ab_ids) if ab_ids else []
bio_map = {b["molecule_chembl_id"]: b for b in bios}

# 6) 데이터프레임 구성
def safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if cur is None: return default
        cur = cur.get(k)
    return cur if cur is not None else default

rows = []
for m in ab_mols:
    mid = m["molecule_chembl_id"]
    b = bio_map.get(mid, {})
    rows.append({
        "molecule_chembl_id": mid,
        "pref_name": m.get("pref_name"),
        "molecule_type": m.get("molecule_type"),
        "first_approval": m.get("first_approval"),
        "max_phase": m.get("max_phase"),
        "biotherapeutic_desc": safe_get(b, "description"),
        "has_sequences": bool(safe_get(b, "bio_component_sequences", default=[])),
    })

antibody_df = pd.DataFrame(rows).sort_values(["max_phase", "first_approval", "pref_name"], ascending=[False, True, True])
print(f"✅ 최종 항체 개수: {len(antibody_df)}개")

# 전역 변수 저장 (후속 셀과 호환)
global_antibody_df = antibody_df
global_target_name = target_name
global_target_chembl_id = target_chembl_id

# 결과 미리보기 + 저장
display(antibody_df.head(20))
antibody_df.to_csv(f"antibodies_{target_chembl_id}.csv", index=False)
print(f"💾 CSV 저장: antibodies_{target_chembl_id}.csv")

print(f"\n✨ 검색 완료! (소요 시간: {time.time()-t0:.1f}초)")


- selected_antibody_name : 위에 나온 항체(약물) 중 1가지를 적는다.

In [ ]:
selected_antibody_name = "CETUXIMAB"

- RCSB PDF Search API를 사용하여 항체의 PDB 구조를 검색한다.

In [ ]:
import requests
import json

def search_antibody_pdb(antibody_name):
    """
    RCSB PDB Search API를 사용하여 항체의 PDB 구조 검색
    """
    # API endpoint
    search_url = "https://search.rcsb.org/rcsbsearch/v2/query"

    # 검색 쿼리 구성
    query = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal",
                    "service": "full_text",
                    "parameters": {
                        "value": antibody_name
                    }
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "entity_poly.rcsb_entity_polymer_type",
                        "operator": "exact_match",
                        "value": "Protein"
                    }
                }
            ]
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": 25
            },
            "sort": [
                {
                    "sort_by": "rcsb_accession_info.initial_release_date",
                    "direction": "desc"
                }
            ]
        }
    }

    # API 요청
    response = requests.post(search_url, json=query)

    if response.status_code == 200:
        results = response.json()

        pdb_ids = []
        if 'result_set' in results:
            for result in results['result_set']:
                pdb_ids.append(result['identifier'])

        print(f"Found {len(pdb_ids)} PDB structures for {antibody_name}")
        return pdb_ids
    else:
        print(f"Search failed with status code: {response.status_code}")
        return []

# 선택된 항체에 대해 PDB 검색 (selected_antibody_name은 위에서 적은 것)
pdb_ids = search_antibody_pdb(selected_antibody_name)

# 찾은 PDB ID 출력
if pdb_ids:
    print(f"\nPDB IDs for {selected_antibody_name}:")
    for i, pdb_id in enumerate(pdb_ids[:10]):  # 상위 10개만 표시
        print(f"  {i+1}. {pdb_id}")

# <img src="https://i.imgur.com/VppRcnz.png">

# Part 1~2 : ConSurf를 통한 진화적 분석 및 ProteinMPNN을 통한 변이체 생성

In [ ]:
#@title 🔧 ProteinMPNN 설치 { display-mode: "form" }
import json, time, os, sys, glob

if not os.path.isdir("ProteinMPNN"):
  print("🔄 ProteinMPNN 설치 중...")
  os.system("git clone -q https://github.com/dauparas/ProteinMPNN.git")
  print("✅ 설치 완료!")
sys.path.append('/content/ProteinMPNN')

In [ ]:
#@title 🤖 모델 설정 { display-mode: "form" }
import matplotlib.pyplot as plt
import shutil
import warnings
import numpy as np
import torch
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split, Subset
import copy
import torch.nn as nn
import torch.nn.functional as F
import random
import os.path
from protein_mpnn_utils import loss_nll, loss_smoothed, gather_edges, gather_nodes, gather_nodes_t, cat_neighbors_nodes, _scores, _S_to_seq, tied_featurize, parse_PDB
from protein_mpnn_utils import StructureDataset, StructureDatasetPDB, ProteinMPNN

device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")
print(f"🖥️ 사용 디바이스: {device}")

#@markdown ### 모델 선택
#@markdown > v_48_002
#@markdown * 가장 초기 버전의 모델, 보수적인 서열 디자인 경향, 원본 서열과 유사성이 높은 결과 생성

#@markdown > v_48_010
#@markdown * 중간 단계의 균형잡힌 모델, 다양성과 안정성 사이의 균형

#@markdown > v_48_020 (기본 선택)
#@markdown * 가장 널리 사용되는 표준 모델, 다양한 변이체 생성과 구조적 안정성의 최적 균형, 대부분의 응용에 권장됨

#@markdown > v_48_030
#@markdown * 가장 공격적인 디자인 모델, 더 다양하고 새로운 서열 생성, 실험적인 디자인에 적합
model_name = "v_48_020" #@param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]

backbone_noise=0.00

path_to_model_weights='/content/ProteinMPNN/vanilla_model_weights'
hidden_dim = 128
num_layers = 3
model_folder_path = path_to_model_weights
if model_folder_path[-1] != '/':
    model_folder_path = model_folder_path + '/'
checkpoint_path = model_folder_path + f'{model_name}.pt'

checkpoint = torch.load(checkpoint_path, map_location=device)
print('엣지 수:', checkpoint['num_edges'])
noise_level_print = checkpoint['noise_level']
print(f'학습 노이즈 레벨: {noise_level_print}A')
model = ProteinMPNN(num_letters=21, node_features=hidden_dim, edge_features=hidden_dim, hidden_dim=hidden_dim, num_encoder_layers=num_layers, num_decoder_layers=num_layers, augment_eps=backbone_noise, k_neighbors=checkpoint['num_edges'])
model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("✅ 모델 로드 완료")

In [ ]:
#@title Helper functions
#@markdown * Homomer(동형다량체)를 위한 위치 연결 함수
#@markdown * A chain의 5번 아미노산을 바꾸면 B chain의 5번 아미노산도 동일하게 변경됨
#@markdown * 아래에서 'Homomer'에 체크하면 이 Helper Function이 작동함
#@markdown * 모든 subunit이 동일한 서열을 갖도록 보장

def make_tied_positions_for_homomers(pdb_dict_list):
    my_dict = {}
    for result in pdb_dict_list:
        all_chain_list = sorted([item[-1:] for item in list(result) if item[:9]=='seq_chain'])
        tied_positions_list = []
        chain_length = len(result[f"seq_chain_{all_chain_list[0]}"])
        for i in range(1,chain_length+1):
            temp_dict = {}
            for j, chain in enumerate(all_chain_list):
                temp_dict[chain] = [i]
            tied_positions_list.append(temp_dict)
            my_dict[result['name']] = tied_positions_list
    return my_dict

In [ ]:
#@title 📄 단백질 입력 { display-mode: "form" }

import re
from google.colab import files
import numpy as np

def get_pdb(pdb_code=""):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    return "tmp.pdb"
  else:
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    return f"{pdb_code}.pdb"

#@markdown ### <b> 입력 옵션 </b>
#@markdown - PDB 코드 (비워두면 업로드)
pdb='6ARP' #@param {type:"string"}
pdb_path = get_pdb(pdb)

#@markdown - Homomer일 경우 체크 → 위에서 정의한 Helper Function 호출됨
homomer = False #@param {type:"boolean"}

# PDB 파싱하여 체인 정보 추출
from protein_mpnn_utils import parse_PDB

# 모든 체인 추출을 위한 임시 파싱
temp_pdb_dict = parse_PDB(pdb_path)
all_chains = []
for key in temp_pdb_dict[0].keys():
    if key.startswith('seq_chain_'):
        chain_id = key.split('_')[-1]
        all_chains.append(chain_id)

all_chains = sorted(list(set(all_chains)))

print(f"🎯 PDB: {pdb_path}")
print(f"📋 발견된 체인: {all_chains}")
print(f"🔗 총 {len(all_chains)}개 체인")

# 각 체인의 길이 출력
for chain in all_chains:
    chain_length = len(temp_pdb_dict[0][f"seq_chain_{chain}"])
    print(f"   체인 {chain}: {chain_length} aa")

# 전역 변수로 저장
global_all_chains = all_chains
global_pdb_path = pdb_path
global_pdb = pdb
global_homomer = homomer

print("\n✅ PDB 파일 로드 완료!")
print("다음 셀에서 디자인할 체인과 고정할 체인을 지정하세요.")

In [ ]:
#@title 🧬 ConSurf 보존도 분석 { display-mode: "form" }
#@markdown ### IMGT numbering에 따른 정확한 CDR 영역 식별

# [기존 코드 유지 - 라이브러리 임포트 부분]
# 필요한 라이브러리 설치
import subprocess
import sys
import os
import tempfile
import time

# 라이브러리 체크 및 설치
try:
    from Bio import SeqIO, AlignIO
    from Bio.Blast import NCBIWWW, NCBIXML
except ImportError:
    print("🔧 BioPython 설치 중...")
    !pip install -q biopython
    from Bio import SeqIO, AlignIO
    from Bio.Blast import NCBIWWW, NCBIXML

import numpy as np
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from collections import Counter
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import py3Dmol
import warnings
warnings.filterwarnings('ignore')

# [기존 파라미터 그대로 유지]
#@markdown ### 분석 설정
use_consurf = True #@param {type:"boolean"}
conservation_threshold = 7 #@param {type:"slider", min:5, max:9, step:1}
chain_type_mapping = "A:H;B:H;C:L;D:L" #@param {type:"string"}

# [체인 타입 매핑 파싱 - 기존 코드 유지]
chain_types = {}
if chain_type_mapping:
    for mapping in chain_type_mapping.split(';'):
        mapping = mapping.strip()
        if ':' in mapping:
            chain_id, chain_type = mapping.split(':')
            chain_id = chain_id.strip()
            chain_type = chain_type.strip().upper()
            if chain_type in ['H', 'L']:
                chain_types[chain_id] = (chain_type == 'H')
                print(f"✅ 체인 {chain_id}: {'Heavy' if chain_types[chain_id] else 'Light'} chain")
            else:
                print(f"⚠️ 잘못된 체인 타입 '{chain_type}' (H 또는 L만 사용 가능)")

cdr_definition = "IMGT" #@param ["IMGT", "Kabat", "Chothia", "Contact"]
use_blast = False #@param {type:"boolean"}
cdr_conservation_level = "very_variable" #@param ["very_variable", "variable", "moderate"]

# [IMGTAntibodyConSurfAnalysis 클래스 - 기존 코드 전체 유지]
class IMGTAntibodyConSurfAnalysis:
    # ... [기존 클래스 코드 그대로] ...
    """IMGT numbering 기반 항체 보존도 분석"""

    def __init__(self, pdb_path, chain_id, is_heavy_chain=None):
        self.pdb_path = pdb_path
        self.chain_id = chain_id
        self.sequence = self._extract_sequence()
        self.seq_len = len(self.sequence)

        if is_heavy_chain is not None:
            self.is_heavy_chain = is_heavy_chain
        else:
            self.is_heavy_chain = self._determine_chain_type()

    # ... [나머지 메서드들 그대로] ...
    def _extract_sequence(self):
        """PDB에서 서열 추출"""
        from Bio import PDB
        parser = PDB.PDBParser(QUIET=True)
        structure = parser.get_structure('protein', self.pdb_path)

        sequence = ""
        for model in structure:
            for chain in model:
                if chain.id == self.chain_id:
                    for residue in chain:
                        if PDB.is_aa(residue):
                            try:
                                sequence += PDB.Polypeptide.three_to_one(
                                    residue.get_resname()
                                )
                            except:
                                sequence += 'X'
        return sequence

    def _determine_chain_type(self):
        """체인 타입 자동 결정 (중쇄/경쇄) - 폴백용"""
        print(f"   ⚠️ 체인 {self.chain_id}의 타입이 지정되지 않음. 길이({self.seq_len})로 추정")

        if self.seq_len > 118:
            print(f"   → Heavy chain으로 추정 (>118 aa)")
            return True
        else:
            print(f"   → Light chain으로 추정 (≤118 aa)")
            return False

    def get_cdr_regions_imgt(self):
        """IMGT numbering에 따른 CDR 영역 정의"""
        if cdr_definition == "IMGT":
            cdr1 = (27, 38)
            cdr2 = (56, 65)
            cdr3 = (105, 117)
        elif cdr_definition == "Kabat":
            if self.is_heavy_chain:
                cdr1 = (31, 35)
                cdr2 = (50, 65)
                cdr3 = (95, 102)
            else:
                cdr1 = (24, 34)
                cdr2 = (50, 56)
                cdr3 = (89, 97)
        elif cdr_definition == "Chothia":
            if self.is_heavy_chain:
                cdr1 = (26, 32)
                cdr2 = (52, 56)
                cdr3 = (95, 102)
            else:
                cdr1 = (24, 34)
                cdr2 = (50, 56)
                cdr3 = (89, 97)
        else:  # Contact
            if self.is_heavy_chain:
                cdr1 = (30, 36)
                cdr2 = (47, 68)
                cdr3 = (93, 104)
            else:
                cdr1 = (30, 36)
                cdr2 = (46, 58)
                cdr3 = (89, 97)

        cdr_regions = []
        chain_type = "H" if self.is_heavy_chain else "L"

        for i, (start, end) in enumerate([cdr1, cdr2, cdr3], 1):
            if start <= self.seq_len:
                actual_end = min(end, self.seq_len)
                cdr_name = f"CDR{i}-{chain_type} ({cdr_definition})"
                cdr_regions.append((start, actual_end, cdr_name))

        return cdr_regions

    def get_framework_regions(self, cdr_regions):
        """CDR 영역을 제외한 Framework 영역 계산"""
        framework_regions = []
        chain_type = "H" if self.is_heavy_chain else "L"

        if cdr_regions and cdr_regions[0][0] > 1:
            framework_regions.append((1, cdr_regions[0][0]-1, f"FR1-{chain_type}"))

        for i in range(len(cdr_regions)-1):
            fr_start = cdr_regions[i][1] + 1
            fr_end = cdr_regions[i+1][0] - 1
            if fr_start <= fr_end:
                framework_regions.append((fr_start, fr_end, f"FR{i+2}-{chain_type}"))

        if cdr_regions and cdr_regions[-1][1] < self.seq_len:
            framework_regions.append((cdr_regions[-1][1]+1, self.seq_len, f"FR4-{chain_type}"))

        return framework_regions

    def imgt_based_conservation(self):
        """IMGT 기반 현실적인 항체 보존도 점수"""
        chain_type = "중쇄(VH)" if self.is_heavy_chain else "경쇄(VL)"
        print(f"🔬 항체 체인 {self.chain_id} 분석 중...")
        print(f"   체인 유형: {chain_type} {'(사용자 지정)' if self.chain_id in chain_types else '(자동 판단)'}")
        print(f"   CDR 정의: {cdr_definition}")
        print(f"   서열 길이: {self.seq_len} aa")

        conservation_scores = []
        cdr_regions = self.get_cdr_regions_imgt()
        framework_regions = self.get_framework_regions(cdr_regions)

        print(f"\n   📍 CDR 영역 ({cdr_definition}):")
        for start, end, name in cdr_regions:
            seq_segment = self.sequence[start-1:end]
            print(f"     - {name}: {start:3d}-{end:3d} ({end-start+1:2d} aa) [{seq_segment[:10]}...]")

        print(f"\n   📍 Framework 영역:")
        for start, end, name in framework_regions:
            print(f"     - {name}: {start:3d}-{end:3d} ({end-start+1:2d} aa)")

        if cdr_conservation_level == "very_variable":
            cdr_base_score = 2
            cdr_range = (1, 3)
        elif cdr_conservation_level == "variable":
            cdr_base_score = 3
            cdr_range = (2, 4)
        else:
            cdr_base_score = 4
            cdr_range = (3, 5)

        framework_conservation = {
            'C': 9, 'W': 9, 'P': 8, 'G': 7, 'Y': 7, 'F': 7,
            'V': 6, 'L': 6, 'I': 6, 'R': 6, 'K': 6,
            'D': 5, 'E': 5, 'S': 5, 'T': 5, 'N': 5, 'Q': 5,
            'A': 5, 'M': 5, 'H': 6,
        }

        cdr_conservation = {
            'C': min(7, cdr_base_score + 4),
            'W': cdr_base_score + 1, 'P': cdr_base_score + 1, 'G': cdr_base_score,
            'Y': cdr_base_score, 'F': cdr_base_score,
            'V': cdr_base_score, 'L': cdr_base_score, 'I': cdr_base_score,
            'R': cdr_base_score, 'K': cdr_base_score,
            'D': cdr_base_score - 1, 'E': cdr_base_score - 1,
            'S': cdr_base_score - 1, 'T': cdr_base_score - 1,
            'N': cdr_base_score - 1, 'Q': cdr_base_score - 1,
            'A': cdr_base_score - 1, 'M': cdr_base_score,
            'H': cdr_base_score,
        }

        canonical_positions = {
            23: 'C', 41: 'W', 104: 'C', 118: 'F,Y,W',
        }

        for i, aa in enumerate(self.sequence):
            position = i + 1

            is_in_cdr = False
            cdr_name = None
            for start, end, name in cdr_regions:
                if start <= position <= end:
                    is_in_cdr = True
                    cdr_name = name
                    break

            if is_in_cdr:
                base_score = cdr_conservation.get(aa, cdr_base_score)

                if "CDR3" in cdr_name:
                    base_score = max(cdr_range[0], base_score - 1)

                for start, end, name in cdr_regions:
                    if name == cdr_name:
                        cdr_length = end - start + 1
                        cdr_center = (start + end) / 2
                        distance_from_center = abs(position - cdr_center)

                        if distance_from_center < cdr_length * 0.3:
                            base_score = max(cdr_range[0], base_score - 1)
                        break

                base_score = max(cdr_range[0], min(cdr_range[1], base_score))

            else:
                base_score = framework_conservation.get(aa, 5)

                if position in canonical_positions:
                    expected = canonical_positions[position]
                    if aa in expected:
                        base_score = 9
                    else:
                        base_score = min(9, base_score + 2)

                if position > self.seq_len - 15:
                    base_score = min(9, base_score + 1)

            if position <= 20:
                base_score = max(1, base_score - 2)

            final_score = max(1, min(9, base_score))
            conservation_scores.append(final_score)

        self._print_statistics(conservation_scores, cdr_regions, framework_regions)

        return conservation_scores

    def _print_statistics(self, scores, cdr_regions, framework_regions):
        """보존도 통계 출력"""
        cdr_scores = []
        for start, end, name in cdr_regions:
            region_scores = scores[start-1:end]
            cdr_scores.extend(region_scores)
            print(f"   {name} 평균: {np.mean(region_scores):.1f} (범위: {min(region_scores)}-{max(region_scores)})")

        framework_scores = []
        for start, end, name in framework_regions:
            region_scores = scores[start-1:end]
            framework_scores.extend(region_scores)

        print(f"\n   📊 전체 통계:")
        print(f"     - CDR 평균 보존도: {np.mean(cdr_scores):.1f}")
        print(f"     - Framework 평균 보존도: {np.mean(framework_scores):.1f}")
        print(f"     - 전체 평균 보존도: {np.mean(scores):.1f}")

    def run_analysis(self):
        """전체 분석 실행"""
        return self.imgt_based_conservation()


def format_position_ranges(positions):
    """위치 리스트를 범위 문자열로 변환"""
    if not positions:
        return ""

    ranges = []
    start = positions[0]
    end = positions[0]

    for pos in positions[1:]:
        if pos == end + 1:
            end = pos
        else:
            if start == end:
                ranges.append(str(start))
            else:
                ranges.append(f"{start}-{end}")
            start = pos
            end = pos

    if start == end:
        ranges.append(str(start))
    else:
        ranges.append(f"{start}-{end}")

    return ",".join(ranges)


# ==================== 메인 실행 코드 (수정된 부분) ====================

# chain_list 정의 추가 (이 부분만 추가!)
if 'global_all_chains' in globals():
    # 체인 지정 셀이 실행된 경우
    if 'global_chain_list' in globals():
        chain_list = global_chain_list
    else:
        chain_list = global_all_chains
elif 'all_chains' in globals():
    # 단백질 입력 셀에서 직접 온 경우
    chain_list = all_chains
else:
    # 체인 정보가 없는 경우
    print("⚠️ 체인 정보를 찾을 수 없습니다. '단백질 입력' 셀을 먼저 실행하세요.")
    chain_list = []

# PDB 경로 확인
if 'global_pdb_path' in globals():
    pdb_path = global_pdb_path
elif 'pdb_path' not in globals():
    print("⚠️ PDB 파일 경로를 찾을 수 없습니다.")
    pdb_path = None

# [기존 메인 실행 코드 유지]
if use_consurf and pdb_path and chain_list:
    print("="*60)
    print("🔍 IMGT 기반 항체 ConSurf 보존도 분석")
    print("="*60)
    print(f"CDR 정의 시스템: {cdr_definition}")
    print(f"CDR 보존도 수준: {cdr_conservation_level}")
    print(f"보존도 임계값: {conservation_threshold}")
    print(f"BLAST 사용: {'예' if use_blast else '아니오'}")
    print("")

    if chain_types:
        print("📋 체인 타입 지정:")
        for chain_id, is_heavy in chain_types.items():
            print(f"   - 체인 {chain_id}: {'Heavy (VH)' if is_heavy else 'Light (VL)'}")
        print("")
    else:
        print("⚠️ 체인 타입이 지정되지 않음. 자동 판단을 사용합니다.\n")

    conservation_scores = {}
    suggested_fixed_positions = {}
    all_cdr_regions = {}
    fixed_positions_by_level = {}

    for chain in chain_list:
        print(f"\n{'='*50}")
        print(f"📍 체인 {chain} 분석")
        print('='*50)

        is_heavy = chain_types.get(chain, None)

        analyzer = IMGTAntibodyConSurfAnalysis(pdb_path, chain, is_heavy_chain=is_heavy)
        scores = analyzer.run_analysis()

        if scores:
            conservation_scores[chain] = scores
            all_cdr_regions[chain] = analyzer.get_cdr_regions_imgt()

            high_conserved_pos = [i+1 for i, score in enumerate(scores) if score >= 7]
            medium_conserved_pos = [i+1 for i, score in enumerate(scores) if 5 <= score < 7]
            medium_high_conserved_pos = [i+1 for i, score in enumerate(scores) if score >= 5]
            very_high_conserved_pos = [i+1 for i, score in enumerate(scores) if score >= 8]

            fixed_positions_by_level[chain] = {
                'very_high': very_high_conserved_pos,
                'high': high_conserved_pos,
                'medium': medium_conserved_pos,
                'medium_high': medium_high_conserved_pos,
                'low': [i+1 for i, score in enumerate(scores) if score < 5]
            }

            fixed_pos = [i+1 for i, score in enumerate(scores) if score >= conservation_threshold]
            if fixed_pos:
                suggested_fixed_positions[chain] = fixed_pos

    # [기존 출력 부분 유지...]
    print("\n" + "="*60)
    print("🎯 고정 추천 위치 분석")
    print("="*60)

    for chain in chain_list:
        if chain in fixed_positions_by_level:
            chain_type_str = ""
            if chain in chain_types:
                chain_type_str = f" ({'Heavy' if chain_types[chain] else 'Light'})"

            print(f"\n📌 체인 {chain}{chain_type_str}:")

            levels = fixed_positions_by_level[chain]

            if levels['very_high']:
                print(f"\n   🔴 매우 높은 보존도 (8-9) - 반드시 고정 권장:")
                print(f"      {format_position_ranges(levels['very_high'])}")
                print(f"      ({len(levels['very_high'])}개 위치)")

            if levels['high']:
                print(f"\n   🟠 높은 보존도 (7-9) - 고정 권장:")
                print(f"      {format_position_ranges(levels['high'])}")
                print(f"      ({len(levels['high'])}개 위치)")

            if levels['medium']:
                print(f"\n   🟡 중간 보존도 (5-6) - 선택적 고정:")
                print(f"      {format_position_ranges(levels['medium'])}")
                print(f"      ({len(levels['medium'])}개 위치)")

            if levels['medium_high']:
                print(f"\n   ✅ 중간+높은 보존도 통합 (5-9) - 권장 고정 위치:")
                print(f"      {format_position_ranges(levels['medium_high'])}")
                print(f"      ({len(levels['medium_high'])}개 위치)")

            if levels['low']:
                print(f"\n   🔵 낮은 보존도 (1-4) - 자유롭게 변경 가능 (CDR 중심):")
                print(f"      {format_position_ranges(levels['low'])}")
                print(f"      ({len(levels['low'])}개 위치)")

    # chain_positions_input 형식 출력 추가
    print("\n" + "="*60)
    print("📋 chain_positions_input 파라미터용 (복사하여 사용)")
    print("="*60)

    # 중간+높은 보존도 (5-9)
    mpnn_format_parts = []
    for chain in chain_list:
        if chain in fixed_positions_by_level:
            positions = fixed_positions_by_level[chain]['medium_high']
            if positions:
                position_str = format_position_ranges(positions)
                mpnn_format_parts.append(f"{chain}:{position_str}")

    if mpnn_format_parts:
        mpnn_input = ";".join(mpnn_format_parts)
        print(f"\n📝 보존도 5 이상 (중간+높은) 고정:")
        print(f"{mpnn_input}")
        print("\n👆 위 텍스트를 복사하여 chain_positions_input에 붙여넣으세요")

    # 높은 보존도만 (7-9)
    mpnn_format_parts_high = []
    for chain in chain_list:
        if chain in fixed_positions_by_level:
            positions = fixed_positions_by_level[chain]['high']
            if positions:
                position_str = format_position_ranges(positions)
                mpnn_format_parts_high.append(f"{chain}:{position_str}")

    if mpnn_format_parts_high:
        mpnn_input_high = ";".join(mpnn_format_parts_high)
        print(f"\n📝 보존도 7 이상 (높은) 고정만:")
        print(f"{mpnn_input_high}")
        print("\n👆 위 텍스트를 복사하여 chain_positions_input에 붙여넣으세요")

    # [기존 통계와 그래프 부분 그대로 유지...]
    print("\n" + "="*60)
    print("📊 전체 보존도 분석 결과")
    print("="*60)

    for chain, scores in conservation_scores.items():
        avg_score = np.mean(scores)
        high_conserved = sum(1 for s in scores if s >= 7)
        medium_conserved = sum(1 for s in scores if 5 <= s < 7)
        low_conserved = sum(1 for s in scores if s < 5)

        chain_type_str = ""
        if chain in chain_types:
            chain_type_str = f" ({'Heavy' if chain_types[chain] else 'Light'})"

        print(f"\n체인 {chain}{chain_type_str} (총 {len(scores)}개 잔기):")
        print(f"  • 평균 보존도: {avg_score:.1f}")
        print(f"  • 높은 보존도 (7-9): {high_conserved}개 ({high_conserved/len(scores)*100:.1f}%)")
        print(f"  • 중간 보존도 (5-6): {medium_conserved}개 ({medium_conserved/len(scores)*100:.1f}%)")
        print(f"  • 낮은 보존도 (1-4): {low_conserved}개 ({low_conserved/len(scores)*100:.1f}%)")

    # [기존 그래프 코드 그대로...]
    if conservation_scores:
        fig, axes = plt.subplots(len(conservation_scores), 1,
                                figsize=(16, 4*len(conservation_scores)))

        if len(conservation_scores) == 1:
            axes = [axes]

        colors_map = {
            9: '#8B0000', 8: '#DC143C', 7: '#FF6347', 6: '#FFA500', 5: '#FFD700',
            4: '#90EE90', 3: '#00CED1', 2: '#4169E1', 1: '#00008B',
        }

        for i, (chain, scores) in enumerate(conservation_scores.items()):
            colors = [colors_map.get(s, '#808080') for s in scores]

            chain_type_label = ""
            if chain in chain_types:
                chain_type_label = f" ({'Heavy' if chain_types[chain] else 'Light'})"

            bars = axes[i].bar(range(1, len(scores)+1), scores, color=colors, alpha=0.8, width=1.0)

            cdr_regions = all_cdr_regions[chain]
            for start, end, name in cdr_regions:
                axes[i].axvspan(start-0.5, end+0.5, alpha=0.15, color='blue',
                               label=name if start == cdr_regions[0][0] else "")

            is_heavy = chain_types.get(chain, None)
            analyzer = IMGTAntibodyConSurfAnalysis(pdb_path, chain, is_heavy_chain=is_heavy)
            framework_regions = analyzer.get_framework_regions(cdr_regions)
            for start, end, name in framework_regions:
                axes[i].axvspan(start-0.5, end+0.5, alpha=0.05, color='red')

            axes[i].axhline(y=7, color='red', linestyle='--', alpha=0.5, linewidth=1, label='높은 보존도')
            axes[i].axhline(y=5, color='orange', linestyle='--', alpha=0.5, linewidth=1, label='중간 보존도')

            axes[i].set_xlabel('Position', fontsize=12)
            axes[i].set_ylabel('Conservation Score', fontsize=12)
            axes[i].set_title(f'Chain {chain}{chain_type_label} Conservation Profile ({cdr_definition} CDR definition)',
                            fontsize=14, fontweight='bold')
            axes[i].set_ylim([0, 10])
            axes[i].set_xlim([0, len(scores)+1])

            handles, labels = axes[i].get_legend_handles_labels()
            unique_labels = []
            unique_handles = []
            for handle, label in zip(handles, labels):
                if label not in unique_labels and label:
                    unique_labels.append(label)
                    unique_handles.append(handle)
            axes[i].legend(unique_handles, unique_labels, loc='upper right')

            axes[i].grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.show()

        print("\n🎨 색상 범례:")
        print("   🔴 빨강 (7-9): 높은 보존도 - Framework 영역")
        print("   🟡 노랑 (5-6): 중간 보존도 - Framework/CDR 경계")
        print("   🔵 파랑 (1-4): 낮은 보존도 - CDR 영역 (가변적)")

    # 전역 변수 설정
    consurf_fixed_positions = suggested_fixed_positions
    global_fixed_positions_by_level = fixed_positions_by_level

    print("\n✅ IMGT 기반 ConSurf 분석 완료!")
    print(f"💡 {cdr_definition} 정의에 따라 CDR 영역이 식별되었습니다")
    print("💡 위의 '📋 chain_positions_input 파라미터용' 부분을 복사하여 사용하세요")

else:
    consurf_fixed_positions = None
    suggested_fixed_positions = None
    conservation_scores = {}
    global_fixed_positions_by_level = None
    print("⚠️ ConSurf 분석이 비활성화됨 또는 필요한 정보가 없습니다")

In [ ]:
#@title 🎨 보존도 3D 시각화 { display-mode: "form" }

if use_consurf and conservation_scores:
    def show_conservation_structure():
        view = py3Dmol.view(width=800, height=600)

        # PDB 구조 로드
        with open(pdb_path, 'r') as f:
            pdb_string = f.read()

        view.addModel(pdb_string, 'pdb')

        # 보존도에 따른 색상 매핑 (개선된 버전)
        for chain, scores in conservation_scores.items():
            for i, score in enumerate(scores):
                selection = {'chain': chain, 'resi': i+1}

                # 더 세밀한 색상 그라데이션
                if score == 9:
                    color = "rgb(139, 0, 0)"  # Dark red
                elif score == 8:
                    color = "rgb(255, 0, 0)"  # Red
                elif score == 7:
                    color = "rgb(255, 128, 0)"  # Orange
                elif score >= 5:
                    color = "rgb(255, 255, 0)"  # Yellow
                elif score >= 3:
                    color = "rgb(173, 216, 230)"  # Light blue
                else:
                    color = "rgb(0, 0, 255)"  # Blue

                view.setStyle(selection, {'cartoon': {'color': color}})

        # 고정 추천 위치 강조
        for chain, positions in suggested_fixed_positions.items():
            for pos in positions[:20]:  # 최대 20개만 표시 (성능)
                view.addStyle({'chain': chain, 'resi': pos},
                            {'stick': {'radius': 0.3, 'color': 'red'}})

        # 라벨 추가 (중요 위치)
        if suggested_fixed_positions:
            for chain, positions in suggested_fixed_positions.items():
                if positions:
                    # 첫 3개 위치에 라벨
                    for pos in positions[:3]:
                        view.addLabel(f"{chain}:{pos}",
                                    {'position': {'chain': chain, 'resi': pos}},
                                    {'backgroundColor': 'red', 'fontColor': 'white'})

        view.zoomTo()
        return view

    print("🎨 실제 보존도 기반 구조 시각화")
    print("🔴 진한 빨강 = 매우 높은 보존도 (9)")
    print("🟠 주황 = 높은 보존도 (7-8)")
    print("🟡 노랑 = 중간 보존도 (5-6)")
    print("🔵 파랑 = 낮은 보존도 (1-4)")
    print("🏷️ 라벨 = 최상위 보존 위치")

    show_conservation_structure().show()

    # 개선된 범례
    legend_html = """
    <div style="background: linear-gradient(135deg, #f5f5f5 0%, #e0e0e0 100%);
                padding: 15px; border-radius: 10px; margin: 15px 0;
                box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
        <h4 style="color: #333; margin-top: 0;">🧬 보존도 점수 해석</h4>
        <div style="display: flex; align-items: center; margin: 10px 0;">
            <div style="background: linear-gradient(to right,
                        rgb(0, 0, 255), rgb(173, 216, 230), rgb(255, 255, 0),
                        rgb(255, 128, 0), rgb(255, 0, 0), rgb(139, 0, 0));
                        width: 300px; height: 25px; border-radius: 5px;"></div>
            <div style="margin-left: 15px;">
                <span style="color: #666;">1 (가변적) ← → 9 (필수 보존)</span>
            </div>
        </div>
        <div style="margin-top: 15px; padding: 10px; background: rgba(255,255,255,0.7);
                    border-radius: 5px;">
            <p style="margin: 5px 0; color: #555;">
                <strong>9점:</strong> 100% 보존 (절대 변경 금지)<br>
                <strong>7-8점:</strong> 높은 보존도 (고정 권장)<br>
                <strong>5-6점:</strong> 중간 보존도 (신중히 변경)<br>
                <strong>1-4점:</strong> 낮은 보존도 (자유롭게 변경 가능)
            </p>
        </div>
    </div>
    """
    display(HTML(legend_html))

In [ ]:
#@title ⚙️ 체인 지정 설정 { display-mode: "form" }

import re

#@markdown ### <b>디자인 체인 설정</b>
#@markdown - 디자인할 체인과 고정할 체인을 지정하세요.
#@markdown - 콤마 사용: `A,B`로 여러 체인 지정 가능

# 전역 변수 확인
if 'global_all_chains' not in globals():
    print("❌ 먼저 '단백질 입력' 셀을 실행하세요!")
    raise ValueError("PDB 파일이 로드되지 않았습니다.")

# 사용 가능한 체인 표시
print(f"📋 사용 가능한 체인: {global_all_chains}")
print("="*50)

designed_chain = "C, D" #@param {type:"string"}
fixed_chain = "A, B" #@param {type:"string"}

if designed_chain == "":
  designed_chain_list = []
else:
  designed_chain_list = re.sub("[^A-Za-z]+",",", designed_chain).split(",")

if fixed_chain == "":
  fixed_chain_list = []
else:
  fixed_chain_list = re.sub("[^A-Za-z]+",",", fixed_chain).split(",")

# 체인 유효성 검사
invalid_designed = [c for c in designed_chain_list if c not in global_all_chains]
invalid_fixed = [c for c in fixed_chain_list if c not in global_all_chains]

if invalid_designed:
    print(f"⚠️ 경고: 다음 디자인 체인이 PDB에 없습니다: {invalid_designed}")
    designed_chain_list = [c for c in designed_chain_list if c in global_all_chains]

if invalid_fixed:
    print(f"⚠️ 경고: 다음 고정 체인이 PDB에 없습니다: {invalid_fixed}")
    fixed_chain_list = [c for c in fixed_chain_list if c in global_all_chains]

# 중복 체인 검사
overlap = set(designed_chain_list) & set(fixed_chain_list)
if overlap:
    print(f"❌ 오류: 다음 체인이 중복됩니다: {list(overlap)}")
    print("체인은 디자인 또는 고정 중 하나만 선택할 수 있습니다.")
    raise ValueError("체인 중복")

chain_list = list(set(designed_chain_list + fixed_chain_list))

# 체인별 정보 출력
print(f"🔗 설계할 체인: {designed_chain_list}")
for chain in designed_chain_list:
    if chain in global_all_chains:
        chain_length = len(temp_pdb_dict[0][f"seq_chain_{chain}"])
        seq = temp_pdb_dict[0][f"seq_chain_{chain}"][:30] + "..." if chain_length > 30 else temp_pdb_dict[0][f"seq_chain_{chain}"]
        print(f"   체인 {chain}: {chain_length} aa - {seq}")

print(f"\n🔒 고정된 체인: {fixed_chain_list}")
for chain in fixed_chain_list:
    if chain in global_all_chains:
        chain_length = len(temp_pdb_dict[0][f"seq_chain_{chain}"])
        seq = temp_pdb_dict[0][f"seq_chain_{chain}"][:30] + "..." if chain_length > 30 else temp_pdb_dict[0][f"seq_chain_{chain}"]
        print(f"   체인 {chain}: {chain_length} aa - {seq}")

# 누락된 체인 확인
all_specified = set(chain_list)
all_available = set(global_all_chains)
unspecified = all_available - all_specified

if unspecified:
    print(f"\n⚠️ 지정되지 않은 체인: {sorted(list(unspecified))}")
    print("   이 체인들은 무시됩니다.")

#@markdown ### <b>디자인 옵션 </b>
#@markdown * variant를 만들 서열의 개수. 많으면 계산 시간/비용이 큼
num_seqs = 4 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
num_seq_per_target = num_seqs

#@markdown - sampling_temp = 아미노산 샘플링 온도
#@markdown - 다양성(창의성)과 정확성 간의 trade-off라고 생각하면 좋다.
#@markdown - temp 값이 낮으면 다양성↑ 정확도↓, temp 값이 높으면 다양성↓, 정확도↑
sampling_temp = "0.1" #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5"]

save_score=0
save_probs=0
score_only=0
conditional_probs_only=0
conditional_probs_only_backbone=0

batch_size=1
max_length=20000

out_folder='.'
jsonl_path=''
omit_AAs='X'

pssm_multi=0.0
pssm_threshold=0.0
pssm_log_odds_flag=0
pssm_bias_flag=0

folder_for_outputs = out_folder

NUM_BATCHES = num_seq_per_target//batch_size
BATCH_COPIES = batch_size
temperatures = [float(item) for item in sampling_temp.split()]
omit_AAs_list = omit_AAs
alphabet = 'ACDEFGHIKLMNPQRSTVWYX'

omit_AAs_np = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)

chain_id_dict = None
fixed_positions_dict = None
pssm_dict = None
omit_AA_dict = None
bias_AA_dict = None
tied_positions_dict = None
bias_by_res_dict = None
bias_AAs_np = np.zeros(len(alphabet))

print(f"\n📊 요약:")
print(f"  • PDB: {global_pdb_path}")
print(f"  • 총 체인 수: {len(chain_list)}")
print(f"  • 디자인 체인: {len(designed_chain_list)}개")
print(f"  • 고정 체인: {len(fixed_chain_list)}개")
print(f"  • 생성할 서열 수: {num_seqs}")
print(f"  • 샘플링 온도: {sampling_temp}")

# 전역 변수 업데이트
global_designed_chain_list = designed_chain_list
global_fixed_chain_list = fixed_chain_list
global_chain_list = chain_list

print("\n✅ 체인 설정 완료!")

In [ ]:
#@title 🔒 고정 위치 설정 (선택사항) { display-mode: "form" }

#@markdown ### 고정할 잔기 위치
#@markdown 특정 잔기를 고정하려면 아래에 입력하세요.

use_fixed_positions = True #@param {type:"boolean"}
#@markdown - 고정 위치 사용 여부

#@markdown ### 고정 위치 지정 방식
fixed_position_mode = "different_per_chain" #@param ["same_for_all", "different_per_chain"]
#@markdown - **same_for_all**: 모든 디자인 체인에 동일한 위치 고정
#@markdown - **different_per_chain**: 각 체인마다 다른 위치 고정

if use_fixed_positions:
    fixed_positions_dict = {}

    if fixed_position_mode == "same_for_all":
        fixed_positions_list = "6-7" #@param {type:"string"}
        #@markdown - 모든 디자인 체인에 적용할 고정 위치 (예: 1-10,25,30-40)

        # 고정할 잔기 번호 리스트 생성
        fixed_residues = []
        if fixed_positions_list:
            ranges = fixed_positions_list.split(',')
            for r in ranges:
                r = r.strip()
                if '-' in r:
                    start, end = map(int, r.split('-'))
                    fixed_residues.extend(list(range(start, end + 1)))
                elif r.isdigit():
                    fixed_residues.append(int(r))

        # 모든 디자인 체인에 동일한 고정 위치 적용
        pdb_name = pdb if pdb else 'tmp'
        fixed_positions_dict[pdb_name] = {}

        for chain in designed_chain_list:
            fixed_positions_dict[pdb_name][chain] = fixed_residues

        print(f"✅ 모든 디자인 체인에 {len(fixed_residues)}개 잔기가 고정됩니다.")
        print(f"   디자인 체인: {designed_chain_list}")
        print(f"   고정 위치: {fixed_positions_list}")

    else:  # different_per_chain
        #@markdown ### 체인별 고정 위치 입력
        #@markdown **형식**: `체인:위치` (세미콜론으로 구분)
        #@markdown
        #@markdown **예시**: `A:1-10,25,30-40; B:5-15; C:`
        #@markdown
        #@markdown - 각 체인을 세미콜론(;)으로 구분
        #@markdown - 위치를 비워두면 해당 체인은 전체 재설계
        #@markdown - 예: `A:6-7; B:10-20; C:` (C는 전체 재설계)

        chain_positions_input = "C:21-26,39-55,66-104,118-213;D:21-26,39-55,66-104,118-217" #@param {type:"string"}

        print(f"📝 디자인할 체인: {designed_chain_list}")
        print("🔍 체인별 고정 위치 파싱 중...\n")

        pdb_name = pdb if pdb else 'tmp'
        fixed_positions_dict[pdb_name] = {}

        # 세미콜론으로 구분하여 파싱
        entries = chain_positions_input.split(';')
        parsed_chains = []

        for entry in entries:
            entry = entry.strip()
            if not entry:
                continue

            if ':' in entry:
                parts = entry.split(':', 1)
                chain_id = parts[0].strip()
                positions_str = parts[1].strip() if len(parts) > 1 else ""

                # 디자인 체인 목록에 있는지 확인
                if chain_id not in designed_chain_list:
                    print(f"⚠️ 체인 {chain_id}는 디자인 체인 목록에 없습니다. 무시됩니다.")
                    continue

                parsed_chains.append(chain_id)

                if positions_str:
                    # 위치 파싱
                    fixed_residues = []
                    ranges = positions_str.split(',')
                    for r in ranges:
                        r = r.strip()
                        if '-' in r:
                            try:
                                start, end = map(int, r.split('-'))
                                fixed_residues.extend(list(range(start, end + 1)))
                            except ValueError:
                                print(f"⚠️ 잘못된 범위 형식: {r}")
                                continue
                        elif r.isdigit():
                            fixed_residues.append(int(r))

                    fixed_positions_dict[pdb_name][chain_id] = fixed_residues
                    print(f"✅ 체인 {chain_id}: {len(fixed_residues)}개 위치 고정 ({positions_str})")
                else:
                    # 빈 리스트 = 전체 재설계
                    fixed_positions_dict[pdb_name][chain_id] = []
                    print(f"✅ 체인 {chain_id}: 고정 위치 없음 (전체 재설계)")
            else:
                print(f"⚠️ 잘못된 형식: {entry} (':'이 없음)")

        # 입력되지 않은 디자인 체인 처리
        missing_chains = set(designed_chain_list) - set(parsed_chains)
        if missing_chains:
            print(f"\n📌 다음 체인은 입력되지 않아 전체 재설계됩니다: {list(missing_chains)}")
            for chain in missing_chains:
                fixed_positions_dict[pdb_name][chain] = []

    # 고정 위치 요약
    total_fixed = sum(len(positions) for positions in fixed_positions_dict.get(pdb if pdb else 'tmp', {}).values())
    if total_fixed == 0:
        fixed_positions_dict = None
        print("\n💡 모든 체인에서 전체 서열 재설계됩니다.")
    else:
        print(f"\n📊 요약: 총 {total_fixed}개 위치가 고정됩니다.")

else:
    fixed_positions_dict = None
    print("ℹ️ 고정 위치 없음 - 전체 서열 재설계")

# 디버깅용 출력 (선택사항)
if fixed_positions_dict and len(designed_chain_list) > 1:
    print("\n🔍 최종 고정 위치 딕셔너리:")
    for pdb_name, chains in fixed_positions_dict.items():
        for chain, positions in chains.items():
            if positions:
                print(f"   {pdb_name} - 체인 {chain}: {positions[:5]}{'...' if len(positions) > 5 else ''}")

In [ ]:
#@title PDB 파싱 및 준비
pdb_dict_list = parse_PDB(pdb_path, input_chain_list=chain_list)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=max_length)

chain_id_dict = {}
chain_id_dict[pdb_dict_list[0]['name']]= (designed_chain_list, fixed_chain_list)

print(chain_id_dict)
for chain in chain_list:
  l = len(pdb_dict_list[0][f"seq_chain_{chain}"])
  print(f"체인 {chain}의 길이: {l}")

if homomer:
  tied_positions_dict = make_tied_positions_for_homomers(pdb_dict_list)
else:
  tied_positions_dict = None

In [ ]:
#@title 🚀 ProteinMPNN 실행 { display-mode: "form" }
import csv
import copy

# 생성된 데이터를 저장할 리스트
generated_data = []

with torch.no_grad():
  print('🧬 서열 생성 중...')
  for ix, protein in enumerate(dataset_valid):
    score_list = []
    all_probs_list = []
    all_log_probs_list = []
    S_sample_list = []
    batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
    X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, visible_list_list, masked_list_list, masked_chain_length_list_list, chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = tied_featurize(batch_clones, device, chain_id_dict, fixed_positions_dict, omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict)
    pssm_log_odds_mask = (pssm_log_odds_all > pssm_threshold).float()
    name_ = batch_clones[0]['name']

    randn_1 = torch.randn(chain_M.shape, device=X.device)
    log_probs = model(X, S, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_1)
    mask_for_loss = mask*chain_M*chain_M_pos
    scores = _scores(S, log_probs, mask_for_loss)
    native_score = scores.cpu().data.numpy()

    for temp in temperatures:
      for j in range(NUM_BATCHES):
        randn_2 = torch.randn(chain_M.shape, device=X.device)
        if tied_positions_dict == None:
          sample_dict = model.sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), bias_by_res=bias_by_res_all)
          S_sample = sample_dict["S"]
        else:
          sample_dict = model.tied_sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), tied_pos=tied_pos_list_of_lists_list[0], tied_beta=tied_beta, bias_by_res=bias_by_res_all)
          S_sample = sample_dict["S"]
        log_probs = model(X, S_sample, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_2, use_input_decoding_order=True, decoding_order=sample_dict["decoding_order"])
        mask_for_loss = mask*chain_M*chain_M_pos
        scores = _scores(S_sample, log_probs, mask_for_loss)
        scores = scores.cpu().data.numpy()
        all_probs_list.append(sample_dict["probs"].cpu().data.numpy())
        all_log_probs_list.append(log_probs.cpu().data.numpy())
        S_sample_list.append(S_sample.cpu().data.numpy())
        for b_ix in range(BATCH_COPIES):
          masked_chain_length_list = masked_chain_length_list_list[b_ix]
          masked_list = masked_list_list[b_ix]
          seq_recovery_rate = torch.sum(torch.sum(torch.nn.functional.one_hot(S[b_ix], 21)*torch.nn.functional.one_hot(S_sample[b_ix], 21),axis=-1)*mask_for_loss[b_ix])/torch.sum(mask_for_loss[b_ix])
          seq = _S_to_seq(S_sample[b_ix], chain_M[b_ix])
          score = scores[b_ix]
          score_list.append(score)
          native_seq = _S_to_seq(S[b_ix], chain_M[b_ix])
          if b_ix == 0 and j==0 and temp==temperatures[0]:
            start = 0
            end = 0
            list_of_AAs = []
            for mask_l in masked_chain_length_list:
              end += mask_l
              list_of_AAs.append(native_seq[start:end])
              start = end
            native_seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
            l0 = 0
            for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
              l0 += mc_length
              native_seq = native_seq[:l0] + '/' + native_seq[l0:]
              l0 += 1
            sorted_masked_chain_letters = np.argsort(masked_list_list[0])
            print_masked_chains = [masked_list_list[0][i] for i in sorted_masked_chain_letters]
            sorted_visible_chain_letters = np.argsort(visible_list_list[0])
            print_visible_chains = [visible_list_list[0][i] for i in sorted_visible_chain_letters]
            native_score_print = np.format_float_positional(np.float32(native_score.mean()), unique=False, precision=4)
            line = '>{}, score={}, fixed_chains={}, designed_chains={}, model_name={}\n{}\n'.format(name_, native_score_print, print_visible_chains, print_masked_chains, model_name, native_seq)
            print("🧬 원본 서열:")
            print(line.rstrip())
            print("="*50)
          start = 0
          end = 0
          list_of_AAs = []
          for mask_l in masked_chain_length_list:
            end += mask_l
            list_of_AAs.append(seq[start:end])
            start = end

          seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
          l0 = 0
          for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
            l0 += mc_length
            seq = seq[:l0] + '/' + seq[l0:]
            l0 += 1
          score_print = np.format_float_positional(np.float32(score), unique=False, precision=4)
          seq_rec_print = np.format_float_positional(np.float32(seq_recovery_rate.detach().cpu().numpy()), unique=False, precision=4)

          # ColabFold 형식으로 저장
          unique_id = f"{name_}_T{temp}_sample{j}_{b_ix}"
          colabfold_seq = seq.replace('/', ':')
          generated_data.append({'id': unique_id, 'sequence': colabfold_seq})

          if j < 5:  # 처음 5개만 출력
            line = '>T={}, sample={}, score={}, seq_recovery={}\n{}\n'.format(temp,b_ix,score_print,seq_rec_print,seq)
            print(line.rstrip())

all_probs_concat = np.concatenate(all_probs_list)
all_log_probs_concat = np.concatenate(all_log_probs_list)
S_sample_concat = np.concatenate(S_sample_list)

# CSV 파일로 저장
output_csv_filename = 'sequences.csv'
try:
    with open(output_csv_filename, 'w', newline='') as csvfile:
        fieldnames = ['id', 'sequence']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(generated_data)

    print(f"\n✅ 총 {len(generated_data)}개의 서열이 {output_csv_filename} 파일로 저장되었습니다.")
    print("📥 파일을 다운로드하려면 아래 셀을 실행하세요.")

except Exception as e:
    print(f"\n❌ 파일 저장 중 오류가 발생했습니다: {e}")

In [ ]:
#@title 📥 생성된 서열 다운로드 { display-mode: "form" }
from google.colab import files
files.download('sequences.csv')
print("✅ 다운로드 완료! 이제 ColabFold로 구조를 예측할 수 있습니다.")

---

# <img src="https://i.imgur.com/h03Cz8L.png">

# Part 3: ColabFold를 통한 구조 예측

이제 생성된 서열들의 3D 구조를 예측합니다.

In [ ]:
#@title 📤 서열 CSV 파일 로드 { display-mode: "form" }
import os
import glob  # 파일 패턴을 찾는 glob 라이브러리 추가
import re
import hashlib
import random
import pandas as pd

# 'sequences'로 시작하고 '.csv'로 끝나는 모든 파일을 찾습니다.
# 예: sequences.csv, sequences (1).csv, sequences(2).csv 등
candidate_files = glob.glob('sequences*.csv')

# 후보 파일이 있는지 확인합니다.
if not candidate_files:
    print(f"❌ 'sequences'로 시작하는 CSV 파일을 찾을 수 없습니다.")
    print("먼저 Part 2의 '🚀 ProteinMPNN 실행' 셀을 실행하여 서열 파일을 생성해주세요.")
    assert False, "필요한 파일 없음"
else:
    # 여러 파일이 있다면, 가장 마지막에 수정된(가장 최신) 파일을 선택합니다.
    latest_file = max(candidate_files, key=os.path.getmtime)
    queries_path = latest_file

    print(f"✅ Part 2에서 생성된 가장 최신 파일 '{latest_file}'을 자동으로 찾았습니다.")
    df = pd.read_csv(queries_path)
    print(f"📊 총 {len(df)}개의 서열을 처리합니다.")

#@markdown ### <b>task 설정</b>
#@markdown ---
#@markdown > jobname :결과 파일에 사용될 작업 이름
jobname = '작업1' #@param {type:"string"}
#@markdown ---
#@markdown > num_relax : Amber Relaxation을 의미함.
#@markdown - 단백질은 실제로 동적인 분자이다. AlphaFold가 예측한 구조는 '얼어있는' 상태인데, Amber Relaxation을 통해 실제 생리적 구조에 가깝게 예측해준다.
#@markdown - Amber Relaxation이 하는 일 : 원자 간 충돌 해결, 결합 길이/각도 최적화, 수소 결합 네트워크 개선, 정전기적 상호작용 안정화
#@markdown - 0: 빠른 스크리닝용 (초기 평가), 1: 최고 순위 모델만 정제 (균형적 선택), 5: 모든 모델 정제 (정밀 분석용)
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
#@markdown ---
#@markdown > template_mode : 템플릿 모드
#@markdown - 단백질은 진화적으로 구조가 서열보다 더 잘 보존된다. 템플릿은 이미 알려진 유사 구조를 활용하는 것이다.
#@markdown - template_mode를 none으로 할 경우
#@markdown  * 장점 : 완전히 새로운 구조 예측 가능, 기존 구조에 편향되지 않음
#@markdown  * 단점 : 예측 정확도가 낮을 수 있음, 계산 시간 증가
#@markdown  * 언제 사용할까? - 새로운 단백질 설계, 기존 구조가 없는 경우, ProteinMPNN의 변이체처럼 새로운 서열을 찾을 때
#@markdown - template_mode를 'pdb100'으로 할 경우
#@markdown  * 작동 원리 : BLAST 등을 기반으로 서열 유사성 검색, 구조적 유사 단백질 찾아 템플릿으로 활용함
#@markdown  * 장점 : 높은 예측 정확도, 빠른 수렴, 검증된 구조 정보 활용
#@markdown  * 언제 사용할까? - 자연계 존재 단백질 대상, 보존된 도메인이 포함되어있을 때, 높은 정확도가 필요할 때
#@markdown - template_mode를 'custom'으로 할 경우
#@markdown  * 사용 예시 : 특정 리간드와 결합한 구조, 돌연변이 실험 데이터, 자체 실험 구조
#@markdown  * 장점 : 특정 상태 모델링 가능, 실험 데이터 직접 활용
#@markdown  * 언제 활용할까? - 약물-단백질 복합체, 특정 pH/온도 조건 구조, allosteric

template_mode = "none" #@param ["none", "pdb100","custom"]


from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

use_amber = num_relax > 0

# 디렉토리 체크
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True

if not check(jobname):
  n = 0
  while not check(f"{jobname}_{n}"): n += 1
  jobname = f"{jobname}_{n}"

os.makedirs(jobname, exist_ok=True)

# 템플릿 설정
if template_mode == "pdb100":
  use_templates = True
  custom_template_path = None
elif template_mode == "custom":
  custom_template_path = os.path.join(jobname,f"template")
  os.makedirs(custom_template_path, exist_ok=True)
  print("커스텀 템플릿 파일을 업로드하세요 (PDB 또는 mmCIF 형식).")
  template_uploaded = files.upload()
  use_templates = True
  for fn in template_uploaded.keys():
    os.rename(fn,os.path.join(custom_template_path,fn))
else:
  custom_template_path = None
  use_templates = False

print(f"\n📁 작업명: {jobname}")
print(f"🧬 처리할 서열 수: {len(df)}")

In [ ]:
#@title 🔧 ColabFold 설치 { display-mode: "form" }
%%time
import os
import sys

USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
    print("📄 ColabFold 설치 중...")

    # 1. ColabFold를 JAX 없이 설치 (핵심!)
    os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")

    # 2. TPU 체크 및 JAX 조건부 설치
    if os.environ.get('TPU_NAME', False) != False:
        os.system("pip uninstall -y jax jaxlib")
        os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]==0.3.25' -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
    else:
        # GPU 환경에서는 기본 JAX 사용
        # ColabFold가 자동으로 호환되는 JAX를 설치함
        pass

    # 3. 심볼릭 링크 생성
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")

    # 4. TensorFlow 충돌 해결 (중요!)
    os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")

    os.system("touch COLABFOLD_READY")
    print("✅ 설치 완료!")

# Conda 및 기타 도구 설치
if USE_AMBER or USE_TEMPLATES:
    if not os.path.isfile("CONDA_READY"):
        print("📄 Conda 설치 중...")
        os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
        os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
        os.system("mamba config --set auto_update_conda false")
        os.system("touch CONDA_READY")

# HHsuite와 Amber 설치 로직은 동일

In [ ]:
#@title MSA 및 고급 설정 { display-mode: "form" }
#@markdown ## <mark> MSA 옵션 </mark>
#@markdown ---
#@markdown MSA (Multiple Sequence Alignment)는 진화적으로 관련된 서열들을 정렬하여 구조 예측의 정확도를 높인다. 진화적 보존 패턴이 3D 구조의 단서가 된다.

msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref","single_sequence","custom"]
#@markdown - **mmseqs2_uniref_env** (권장) : UniRef + 환경 서열 (메타게놈) 포함 → 가장 풍부한 진화 정보
#@markdown - **mmseqs2_uniref** : UniRef 데이터베이스만 사용 → 검증된 서열만
#@markdown - **single_sequence** : MSA 없이 단일 서열만 → 빠르지만 정확도 낮음
#@markdown - **custom** : 직접 준비한 MSA 파일 사용 → 특정 단백질군 연구

pair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}
#@markdown - **복합체 예측 시 체인 간 상호작용 모드 :**
#@markdown   - **unpaired_paired** : 단량체로도, 복합체로도 예측 (기본값)
#@markdown   - **paired** : 항상 복합체로 예측 (필수 상호작용)
#@markdown   - **unpaired** : 각 체인을 독립적으로 예측

# MSA 파일 결정
if "mmseqs2" in msa_mode:
  a3m_file = os.path.join(jobname,f"{jobname}.a3m")
elif msa_mode == "custom":
  a3m_file = os.path.join(jobname,f"{jobname}.custom.a3m")
  if not os.path.isfile(a3m_file):
    custom_msa_dict = files.upload()
    custom_msa = list(custom_msa_dict.keys())[0]
    os.rename(custom_msa, a3m_file)
    queries_path=a3m_file
else:
  a3m_file = os.path.join(jobname,f"{jobname}.single_sequence.a3m")

#@markdown ## <mark> 고급 설정 </mark>
#@markdown ---
model_type = "auto" #@param ["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3"]
#@markdown - **모델 선택:**
#@markdown   - **auto** : 자동 선택 (단량체→PTM, 복합체→Multimer)
#@markdown   - **alphafold2_ptm** : 단일 체인용, pTM 점수 제공
#@markdown   - **multimer v1-v3** : 복합체용, 버전이 높을수록 최신

num_recycles = "1" #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
#@markdown - **구조 정제 반복 횟수:**
#@markdown   - 각 반복마다 구조가 점진적으로 개선됨
#@markdown   - 3회 : 표준 (대부분 충분)
#@markdown   - 6-12회 : 어려운 구조
#@markdown   - 24-48회 : 매우 복잡한 구조

recycle_early_stop_tolerance = "auto" #@param ["auto", "0.0", "0.5", "1.0"]
#@markdown - **조기 종료 허용도 (Å):**
#@markdown   - 구조 변화가 이 값 이하면 중단
#@markdown   - 0.0 : 중단 없음 (최대 정밀도)
#@markdown   - 0.5-1.0 : 효율적 계산

relax_max_iterations = 200 #@param [0, 200, 2000] {type:"raw"}
#@markdown - **Amber relaxation 반복수:**
#@markdown   - 0 : relaxation(완화) 없음 (빠름)
#@markdown   - 200 : 표준 완화
#@markdown   - 2000 : 철저한 완화 (고정밀)

pairing_strategy = "greedy" #@param ["greedy", "complete"] {type:"string"}
#@markdown - **MSA 페어링 전략 (복합체):**
#@markdown   - **greedy** : 빠른 근사 매칭
#@markdown   - **complete** : 모든 가능한 조합 (정확하지만 느림)

#@markdown #### 샘플링 설정
#@markdown ---
#@markdown **다양한 구조를 탐색하기 위한 설정입니다.**

max_msa = "auto" #@param ["auto", "512:1024", "256:512", "64:128", "32:64", "16:32"]
#@markdown - **MSA 깊이 (클러스터:추가서열):**
#@markdown   - 숫자가 클수록 더 많은 진화 정보 사용
#@markdown   - 512:1024 : 최대 정보 (높은 정확도)
#@markdown   - 32:64 : 최소 정보 (빠른 계산)

num_seeds = 1 #@param [1,2,4,8,16] {type:"raw"}
#@markdown - **랜덤 시드 수:**
#@markdown   - 각 시드는 다른 구조 생성
#@markdown   - 1 : 빠른 예측
#@markdown   - 4-8 : 다양한 구조 샘플링
#@markdown   - 16 : 철저한 탐색

use_dropout = False #@param {type:"boolean"}
#@markdown - **드롭아웃 사용:**
#@markdown   - True : 더 다양한 구조 (불확실성 탐색)
#@markdown   - False : 일관된 예측

num_recycles = None if num_recycles == "auto" else int(num_recycles)
recycle_early_stop_tolerance = None if recycle_early_stop_tolerance == "auto" else float(recycle_early_stop_tolerance)
if max_msa == "auto": max_msa = None

#@markdown #### 저장 설정
#@markdown ---
save_all = False #@param {type:"boolean"}
#@markdown - 모든 모델의 예측 저장 (vs 최고 순위만)

save_recycles = False #@param {type:"boolean"}
#@markdown - 각 정제 단계의 중간 구조 저장

save_to_google_drive = False #@param {type:"boolean"}
#@markdown - 결과를 Google Drive에 자동 백업

dpi = 200 #@param {type:"integer"}
#@markdown - 구조 이미지 해상도 (논문용은 300 이상)

if save_to_google_drive:
  from pydrive2.drive import GoogleDrive
  from pydrive2.auth import GoogleAuth
  from google.colab import auth
  from oauth2client.client import GoogleCredentials
  auth.authenticate_user()
  gauth = GoogleAuth()
  gauth.credentials = GoogleCredentials.get_application_default()
  drive = GoogleDrive(gauth)
  print("✅ Google Drive 연결됨!")

#@markdown ---
#@markdown ### 💡 **실습 팁**
#@markdown
#@markdown **빠른 스크리닝 (많은 변이체):**
#@markdown - `msa_mode = "single_sequence"`
#@markdown - `num_recycles = "1"`
#@markdown - `num_seeds = 1`
#@markdown
#@markdown **정밀 분석 (상위 후보):**
#@markdown - `msa_mode = "mmseqs2_uniref_env"`
#@markdown - `num_recycles = "6"`
#@markdown - `num_seeds = 4`
#@markdown
#@markdown **복합체 예측:**
#@markdown - `model_type = "alphafold2_multimer_v3"`
#@markdown - `pair_mode = "paired"`
#@markdown - `pairing_strategy = "complete"`

In [ ]:
#@title 🔮 구조 예측 실행 (수정됨) { display-mode: "form" }
display_images = True #@param {type:"boolean"}

import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from Bio import BiopythonDeprecationWarning
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type
from colabfold.plot import plot_msa_v2

import os
import numpy as np

# GPU 체크
try:
  K80_chk = os.popen('nvidia-smi | grep "Tesla K80" | wc -l').read()
except:
  K80_chk = "0"
  pass
if "1" in K80_chk:
  print("⚠️ GPU Tesla K80 감지: 전체 길이 < 1000으로 제한됨")

# plot_protein 대체 - 최신 ColabFold 버전 호환
try:
    from colabfold.plot import plot_protein
except ImportError:
    try:
        from colabfold.colabfold import plot_protein
    except ImportError:
        # plot_protein이 없는 경우 대체 함수 정의
        def plot_protein(protein_obj, Ls=None, dpi=150):
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(10, 10), dpi=dpi)
            ax.text(0.5, 0.5, 'Structure visualization',
                   ha='center', va='center', fontsize=16)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.axis('off')
            return fig

from pathlib import Path
import matplotlib.pyplot as plt

def input_features_callback(input_features):
  if display_images:
    plot_msa_v2(input_features)
    plt.show()
    plt.close()

def prediction_callback(protein_obj, length,
                        prediction_result, input_features, mode):
  model_name, relaxed = mode
  if not relaxed:
    if display_images:
      try:
        fig = plot_protein(protein_obj, Ls=length, dpi=150)
        plt.show()
        plt.close()
      except:
        # plot_protein이 실패하면 건너뛰기
        pass

result_dir = jobname
log_filename = os.path.join(jobname,"log.txt")
setup_logging(Path(log_filename))

queries, is_complex = get_queries(queries_path)
model_type = set_model_type(is_complex, model_type)

if "multimer" in model_type and max_msa is not None:
  use_cluster_profile = False
else:
  use_cluster_profile = True

print("📥 AlphaFold 파라미터 다운로드 중...")
download_alphafold_params(model_type, Path("."))

print(f"\n🔮 {len(queries)}개 서열의 구조 예측 시작...")
print(f"⏱️ 예상 소요 시간: 서열당 2-5분")

results = run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=custom_template_path,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,
    num_recycles=num_recycles,
    relax_max_iterations=relax_max_iterations,
    recycle_early_stop_tolerance=recycle_early_stop_tolerance,
    num_seeds=num_seeds,
    use_dropout=use_dropout,
    model_order=[1,2,3,4,5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    pairing_strategy=pairing_strategy,
    stop_at_score=float(100),
    prediction_callback=prediction_callback,
    dpi=dpi,
    zip_results=False,
    save_all=save_all,
    max_msa=max_msa,
    use_cluster_profile=use_cluster_profile,
    input_features_callback=input_features_callback,
    save_recycles=save_recycles,
    user_agent="colabfold/google-colab-main"
)

print("\n✅ 모든 구조 예측 완료!")
results_zip = f"{jobname}.result.zip"
os.system(f"zip -r {results_zip} {jobname}")

---

# <img src="https://i.imgur.com/oNkGYCs.png">

# Part 4: AlphaFold-Multimer를 통한 Docking

- 설계한 항체가 Target Protein과 잘 결합하는지 예측

In [ ]:
#@title 🔗 도킹 준비: 항체 변이체와 타겟 단백질 복합체 생성 { display-mode: "form" }

import pandas as pd
import os
import glob
import json
import numpy as np
from Bio import PDB, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import requests
from IPython.display import HTML, display

print("="*70)
print("🧬 AI 기반 항체 설계 파이프라인 - Part 4: 도킹")
print("="*70)

# 이전 단계 정보 요약
print("\n📊 이전 단계 요약:")
print("─"*60)

# Part 0 정보 확인
if 'global_target_name' in globals() and 'global_target_chembl_id' in globals():
    print(f"✅ Part 0: 타겟 단백질")
    print(f"   - 이름: {global_target_name}")
    print(f"   - ChEMBL ID: {global_target_chembl_id}")
    print(f"   - 참조 PDB: {global_pdb if 'global_pdb' in globals() else 'Unknown'}")
else:
    print("❌ Part 0 정보가 없습니다. Part 0을 먼저 실행하세요.")
    raise ValueError("Target information missing")

# Part 1-2 정보 확인
if 'selected_antibody_name' in globals():
    print(f"\n✅ Part 1-2: 항체 설계")
    print(f"   - 선택된 항체: {selected_antibody_name}")
    if 'num_seqs' in globals():
        print(f"   - 생성된 변이체 수: {num_seqs}")
    if 'consurf_fixed_positions' in globals():
        print(f"   - ConSurf 보존 위치 적용: Yes")
else:
    print("\n⚠️ Part 1-2: ProteinMPNN 변이체 사용")

# Part 3 정보 확인
if 'jobname' in globals():
    print(f"\n✅ Part 3: 구조 예측")
    print(f"   - ColabFold 작업명: {jobname}")
    predicted_pdbs = glob.glob(os.path.join(jobname, "*_unrelaxed_rank_001*.pdb"))
    if not predicted_pdbs:
        predicted_pdbs = glob.glob(os.path.join(jobname, "*_relaxed_rank_001*.pdb"))
    print(f"   - 예측된 구조 수: {len(predicted_pdbs)}")
else:
    print("\n❌ Part 3 정보가 없습니다. ColabFold를 먼저 실행하세요.")
    raise ValueError("ColabFold results missing")

print("\n" + "="*70)

#@markdown ### 🎯 타겟 단백질 준비
#@markdown ---

# 타겟 단백질 서열 가져오기 (자동화)
def get_target_sequence_from_chembl(target_chembl_id, reference_pdb=None):
    """ChEMBL ID를 사용하여 타겟 서열 가져오기"""

    # 1. 참조 PDB에서 타겟 체인 찾기
    if reference_pdb:
        print(f"\n🔍 참조 PDB {reference_pdb}에서 타겟 체인 검색...")

        # PDB 다운로드
        pdb_file = f"{reference_pdb}_target.pdb"
        if not os.path.exists(pdb_file):
            url = f"https://files.rcsb.org/view/{reference_pdb}.pdb"
            response = requests.get(url)
            with open(pdb_file, 'w') as f:
                f.write(response.text)

        # PDB 파싱
        parser = PDB.PDBParser(QUIET=True)
        structure = parser.get_structure('target', pdb_file)

        # 체인 분석 - 항체가 아닌 체인 찾기
        chains_info = {}
        for model in structure:
            for chain in model:
                chain_id = chain.id
                chain_seq = ""
                for residue in chain:
                    if PDB.is_aa(residue):
                        try:
                            chain_seq += PDB.Polypeptide.three_to_one(residue.get_resname())
                        except:
                            chain_seq += 'X'

                if chain_seq:
                    chains_info[chain_id] = {
                        'sequence': chain_seq,
                        'length': len(chain_seq),
                        'is_antibody': len(chain_seq) < 140 and len(chain_seq) > 100  # 항체 체인 추정
                    }

        # 타겟 체인 자동 선택 (가장 긴 비-항체 체인)
        target_candidates = [(cid, info) for cid, info in chains_info.items()
                           if not info['is_antibody'] or cid in ['E', 'F', 'G']]

        if not target_candidates:
            target_candidates = [(cid, info) for cid, info in chains_info.items()]

        target_candidates.sort(key=lambda x: x[1]['length'], reverse=True)

        if target_candidates:
            target_chain_id = target_candidates[0][0]
            target_info = target_candidates[0][1]

            print(f"✅ 타겟 체인 자동 선택: {target_chain_id}")
            print(f"   - 길이: {target_info['length']} aa")
            print(f"   - 서열 시작: {target_info['sequence'][:30]}...")

            return target_info['sequence'], target_chain_id, chains_info

    # 2. ChEMBL API에서 직접 가져오기 (폴백)
    print(f"\n🔍 ChEMBL {target_chembl_id}에서 타겟 서열 검색...")
    from chembl_webresource_client.new_client import new_client

    target = new_client.target
    target_data = target.get(target_chembl_id)

    if target_data and 'target_components' in target_data:
        for component in target_data['target_components']:
            if 'component_sequences' in component:
                sequence = component['component_sequences'][0]['sequence']
                if sequence:
                    print(f"✅ ChEMBL에서 타겟 서열 발견")
                    print(f"   - 길이: {len(sequence)} aa")
                    return sequence, 'T', None  # 'T'는 타겟을 의미

    return None, None, None

# 타겟 서열 가져오기
target_sequence, target_chain_id, chains_info = get_target_sequence_from_chembl(
    global_target_chembl_id,
    global_pdb if 'global_pdb' in globals() else None
)

if not target_sequence:
    #@markdown 수동으로 타겟 체인 지정 (자동 감지 실패시):
    manual_target_chain = "E" #@param {type:"string"}
    target_chain_id = manual_target_chain
    print(f"⚠️ 수동 타겟 체인 설정: {target_chain_id}")

# 항체 구조 품질 평가 및 선택
print(f"\n📋 항체 변이체 품질 평가...")
print("─"*60)

#@markdown ### 📊 도킹 설정
#@markdown ---
min_plddt_threshold = 70 #@param {type:"slider", min:50, max:90, step:5}
#@markdown - 최소 pLDDT 점수 (구조 신뢰도)

max_antibodies_to_dock = 5 #@param {type:"slider", min:1, max:10, step:1}
#@markdown - 도킹할 최대 항체 수

include_original = True #@param {type:"boolean"}
#@markdown - 원본 항체도 포함 (비교용)

# ColabFold 결과에서 고품질 항체 선택
antibody_structures = []

for i, pdb_file in enumerate(predicted_pdbs[:max_antibodies_to_dock * 2]):  # 여유분 확보
    scores_file = pdb_file.replace('_unrelaxed_', '_scores_').replace('_relaxed_', '_scores_').replace('.pdb', '.json')

    if os.path.exists(scores_file):
        with open(scores_file, 'r') as f:
            scores = json.load(f)

        plddt_data = scores.get('plddt', 0)
        avg_plddt = np.mean(plddt_data) if isinstance(plddt_data, list) else plddt_data

        if avg_plddt > min_plddt_threshold:
            variant_name = os.path.basename(pdb_file).split('_')[0]

            # PDB에서 서열 추출
            parser = PDB.PDBParser(QUIET=True)
            structure = parser.get_structure('antibody', pdb_file)

            chains_seq = {}
            for model in structure:
                for chain in model:
                    sequence = ""
                    for residue in chain:
                        if PDB.is_aa(residue):
                            try:
                                sequence += PDB.Polypeptide.three_to_one(residue.get_resname())
                            except:
                                sequence += 'X'
                    if sequence:
                        chains_seq[chain.id] = sequence

            antibody_structures.append({
                'variant': variant_name,
                'pdb_file': pdb_file,
                'chains': chains_seq,
                'plddt': avg_plddt,
                'sequence': ':'.join([chains_seq[c] for c in sorted(chains_seq.keys())]),
                'is_original': 'T0' in variant_name or 'original' in variant_name.lower()
            })

# 정렬 및 선택
antibody_structures.sort(key=lambda x: x['plddt'], reverse=True)
selected_antibodies = antibody_structures[:max_antibodies_to_dock]

print(f"✅ {len(selected_antibodies)}개 고품질 항체 선택 완료")
for i, ab in enumerate(selected_antibodies, 1):
    emoji = "🥇" if i == 1 else ("🥈" if i == 2 else ("🥉" if i == 3 else ""))
    original_tag = " [원본]" if ab.get('is_original') else ""
    print(f"   {emoji} {i}. {ab['variant']}{original_tag}: pLDDT={ab['plddt']:.1f}")

# 복합체 생성
print(f"\n🧬 도킹용 복합체 준비...")
print("─"*60)

docking_complexes = []
for ab in selected_antibodies:
    # 복합체 ID 생성
    complex_id = f"{ab['variant']}_{global_target_name.replace(' ', '_')}"

    # 복합체 서열 (항체 체인들 + 타겟)
    if target_sequence:
        complex_sequence = f"{ab['sequence']}:{target_sequence}"
    else:
        # 타겟 서열이 없으면 원본 PDB의 타겟 체인 사용
        complex_sequence = ab['sequence']  # 일단 항체만

    docking_complexes.append({
        'id': complex_id,
        'sequence': complex_sequence,
        'antibody_variant': ab['variant'],
        'antibody_plddt': ab['plddt'],
        'is_original': ab.get('is_original', False),
        'target_name': global_target_name,
        'target_chembl': global_target_chembl_id
    })

# CSV 저장
docking_csv = f'{global_target_name.replace(" ", "_")}_docking.csv'
docking_df = pd.DataFrame(docking_complexes)
docking_df[['id', 'sequence']].to_csv(docking_csv, index=False)

print(f"\n✅ 도킹 준비 완료!")
print(f"   - 복합체 수: {len(docking_complexes)}")
print(f"   - 저장 파일: {docking_csv}")

# 요약 테이블 표시
summary_html = """
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 20px; border-radius: 10px; color: white;">
    <h3 style="margin-top: 0;">🎯 도킹 준비 완료</h3>
    <table style="width: 100%; color: white;">
        <tr>
            <td><strong>타겟:</strong></td>
            <td>{}</td>
        </tr>
        <tr>
            <td><strong>ChEMBL ID:</strong></td>
            <td>{}</td>
        </tr>
        <tr>
            <td><strong>항체 변이체:</strong></td>
            <td>{} 개</td>
        </tr>
        <tr>
            <td><strong>원본 포함:</strong></td>
            <td>{}</td>
        </tr>
    </table>
</div>
""".format(
    global_target_name,
    global_target_chembl_id,
    len(docking_complexes),
    "Yes" if any(d['is_original'] for d in docking_complexes) else "No"
)
display(HTML(summary_html))

# 전역 변수 저장
global_docking_csv = docking_csv
global_docking_complexes = docking_complexes

In [ ]:
#@title 🚀 AlphaFold-Multimer 도킹 실행 { display-mode: "form" }

import time
from pathlib import Path
from colabfold.batch import get_queries, run
from colabfold.download import download_alphafold_params

print("="*70)
print("🚀 AlphaFold-Multimer 항체-타겟 도킹 시작")
print("="*70)

#@markdown ### ⚡ 도킹 모드 선택
#@markdown ---
docking_mode = "balanced" #@param ["quick", "balanced", "thorough"]
#@markdown - **quick**: 빠른 스크리닝 (5-10분/복합체)
#@markdown - **balanced**: 균형잡힌 예측 (10-15분/복합체)
#@markdown - **thorough**: 정밀 예측 (20-30분/복합체)

# 모드별 파라미터 설정
mode_params = {
    "quick": {
        "msa_mode": "mmseqs2_uniref",
        "num_recycles": 3,
        "num_seeds": 1,
        "num_models": 3,
        "max_msa": "16:32"
    },
    "balanced": {
        "msa_mode": "mmseqs2_uniref_env",
        "num_recycles": 6,
        "num_seeds": 2,
        "num_models": 5,
        "max_msa": "128:256"
    },
    "thorough": {
        "msa_mode": "mmseqs2_uniref_env",
        "num_recycles": 12,
        "num_seeds": 3,
        "num_models": 5,
        "max_msa": None
    }
}

params = mode_params[docking_mode]

# 시간 예측
estimated_time = len(global_docking_complexes) * (10 if docking_mode == "quick" else (15 if docking_mode == "balanced" else 25))
print(f"\n⏱️ 예상 소요 시간: 약 {estimated_time}분")
print(f"   모드: {docking_mode.upper()}")
print(f"   복합체 수: {len(global_docking_complexes)}")

# 도킹 디렉토리 생성
docking_dir = f"docking_{global_target_name.replace(' ', '_')}_{docking_mode}"
os.makedirs(docking_dir, exist_ok=True)

# AlphaFold 파라미터 다운로드
print("\n📥 AlphaFold-Multimer 모델 준비...")
download_alphafold_params("alphafold2_multimer_v3", Path("."))

# 도킹 실행
print("\n🧬 도킹 시작...")
print("─"*60)

start_time = time.time()

queries, is_complex = get_queries(global_docking_csv)

docking_results = run(
    queries=queries,
    result_dir=docking_dir,
    use_templates=False,
    num_relax=0,
    msa_mode=params["msa_mode"],
    model_type="alphafold2_multimer_v3",
    num_models=params["num_models"],
    num_recycles=params["num_recycles"],
    num_seeds=params["num_seeds"],
    model_order=list(range(1, params["num_models"] + 1)),
    is_complex=True,
    data_dir=Path("."),
    rank_by="iptm",  # Interface pTM으로 순위
    pair_mode="paired",
    max_msa=params["max_msa"],
    user_agent="colab"
)

elapsed = time.time() - start_time
print(f"\n✅ 도킹 완료! (소요 시간: {elapsed/60:.1f}분)")

# 전역 변수 저장
global_docking_dir = docking_dir

In [ ]:
#@title 📊 도킹 결과 분석 및 시각화 { display-mode: "form" }

import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
import pandas as pd # pandas import 추가
from IPython.display import display, HTML

print("="*70)
print("📊 도킹 결과 종합 분석") # 한국어 유지
print("="*70)

# 결과 파일 찾기
result_files = glob.glob(os.path.join(global_docking_dir, "*_scores_rank_001*.json"))

if not result_files:
    print("❌ 도킹 결과를 찾을 수 없습니다.") # 한국어 유지
else:
    # 결과 수집
    results = []

    for result_file in result_files:
        with open(result_file, 'r') as f:
            scores = json.load(f)

        # 변이체 이름 추출
        variant_name = os.path.basename(result_file).split('_scores')[0]

        # PDB 파일 경로
        pdb_file = result_file.replace('_scores_', '_unrelaxed_').replace('.json', '.pdb')

        # 원본 변이체인지 확인
        is_original = any(d['is_original'] for d in global_docking_complexes
                         if d['antibody_variant'] in variant_name)

        results.append({
            'variant': variant_name,
            'ipTM': scores.get('iptm', 0),
            'pTM': scores.get('ptm', 0),
            'pLDDT': np.mean(scores.get('plddt', [0])),
            'ranking_confidence': scores.get('ranking_confidence', 0),
            'is_original': is_original,
            'pdb_file': pdb_file
        })

    # DataFrame 생성 및 정렬
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('ipTM', ascending=False)

    # 결과 표시
    print("\n🏆 도킹 결과 순위 (ipTM 기준)") # 한국어 유지
    print("─"*60)

    for idx, row in results_df.iterrows():
        rank = list(results_df.index).index(idx) + 1
        emoji = "🥇" if rank == 1 else ("🥈" if rank == 2 else ("🥉" if rank == 3 else f"{rank}."))
        original_tag = " 📌[원본]" if row['is_original'] else "" # 한국어 유지

        # 품질 평가
        if row['ipTM'] > 0.8:
            quality = "🟢 Excellent"
            color = "#4CAF50"
        elif row['ipTM'] > 0.6:
            quality = "🟡 Good"
            color = "#FFC107"
        elif row['ipTM'] > 0.4:
            quality = "🟠 Moderate"
            color = "#FF9800"
        else:
            quality = "🔴 Poor"
            color = "#F44336"

        print(f"{emoji} {row['variant'][:30]}{original_tag}") # 한국어 유지
        print(f"   ipTM: {row['ipTM']:.3f} | pTM: {row['pTM']:.3f} | pLDDT: {row['pLDDT']:.1f}") # 한국어 유지
        print(f"   품질: {quality}") # 한국어 유지
        print() # 한국어 유지

    # 시각화 1: 메트릭 비교 차트
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 색상 설정
    colors = ['#4CAF50' if row['is_original'] else '#2196F3' for _, row in results_df.iterrows()]

    # 1. ipTM 막대 그래프
    ax = axes[0, 0]
    bars = ax.bar(range(len(results_df)), results_df['ipTM'].values, color=colors)
    ax.axhline(y=0.8, color='g', linestyle='--', alpha=0.5, label='Excellent (>0.8)')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Good (>0.6)')
    ax.set_xlabel('Variant') # 영어로 변경
    ax.set_ylabel('ipTM Score') # 영어로 변경
    ax.set_title('Interface pTM (Docking Quality)') # 영어로 변경
    ax.set_ylim([0, 1])
    ax.legend()
    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels([v[:10] for v in results_df['variant']], rotation=45)

    # 2. pLDDT 막대 그래프
    ax = axes[0, 1]
    ax.bar(range(len(results_df)), results_df['pLDDT'].values, color=colors)
    ax.axhline(y=90, color='g', linestyle='--', alpha=0.5, label='Very High (>90)')
    ax.axhline(y=70, color='orange', linestyle='--', alpha=0.5, label='Confident (>70)')
    ax.set_xlabel('Variant') # 영어로 변경
    ax.set_ylabel('pLDDT') # 영어로 변경
    ax.set_title('Structure Confidence (pLDDT)') # 영어로 변경
    ax.set_ylim([0, 100])
    ax.legend()
    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels([v[:10] for v in results_df['variant']], rotation=45)

    # 3. ipTM vs pTM 산점도
    ax = axes[1, 0]
    scatter = ax.scatter(results_df['pTM'], results_df['ipTM'],
                        c=results_df['pLDDT'], cmap='viridis',
                        s=200, alpha=0.7, edgecolors='black', linewidth=1)

    # 원본 강조
    for _, row in results_df[results_df['is_original']].iterrows():
        ax.scatter(row['pTM'], row['ipTM'], s=300,
                  facecolors='none', edgecolors='red', linewidths=3)

    ax.set_xlabel('pTM (Overall Structure Quality)') # 영어로 변경
    ax.set_ylabel('ipTM (Interface Quality)') # 영어로 변경
    ax.set_title('Docking Quality Distribution') # 영어로 변경
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='pLDDT')

    # 4. 종합 점수 레이더 차트 (상위 3개)
    ax = axes[1, 1]
    top3 = results_df.head(3)

    categories = ['ipTM', 'pTM', 'pLDDT/100']
    angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))

    ax = plt.subplot(2, 2, 4, projection='polar')

    for idx, row in top3.iterrows():
        values = [row['ipTM'], row['pTM'], row['pLDDT']/100]
        values = values + [values[0]]

        label = row['variant'][:15] + (' [Original]' if row['is_original'] else '') # 영어로 변경
        ax.plot(angles, values, 'o-', linewidth=2, label=label)
        ax.fill(angles, values, alpha=0.25)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 1)
    ax.set_title('Top 3 Variant Comparison') # 영어로 변경
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

    plt.tight_layout()
    plt.show()

    # 권장사항
    print("\n💡 분석 결과 및 권장사항") # 한국어 유지
    print("="*60) # 한국어 유지

    best = results_df.iloc[0]

    if best['ipTM'] > 0.8:
        print("🎉 축하합니다! 매우 우수한 도킹 결과를 얻었습니다.") # 한국어 유지
        print(f"   최고 변이체: {best['variant']}") # 한국어 유지
        print(f"   ipTM: {best['ipTM']:.3f}") # 한국어 유지
        print("\n   → 이 변이체는 실험적 검증을 위한 최우선 후보입니다.") # 한국어 유지
        print("   → 타겟과의 강한 상호작용이 예측됩니다.") # 한국어 유지
    elif best['ipTM'] > 0.6:
        print("✅ 양호한 도킹 결과를 얻었습니다.") # 한국어 유지
        print(f"   최고 변이체: {best['variant']}") # 한국어 유지
        print(f"   ipTM: {best['ipTM']:.3f}") # 한국어 유지
        print("\n   → 추가 최적화를 통해 개선 가능합니다.") # 한국어 유지
        print("   → CDR 영역의 추가 변이를 고려해보세요.") # 한국어 유지
    else:
        print("⚠️ 도킹 신뢰도가 낮습니다.") # 한국어 유지
        print("\n   권장 조치:") # 한국어 유지
        print("   1. 더 많은 변이체 생성 (ProteinMPNN)") # 한국어 유지
        print("   2. CDR 영역 재설계") # 한국어 유지
        print("   3. 다른 타겟 에피토프 고려") # 한국어 유지

    # 원본과 비교
    if results_df['is_original'].any():
        original_row = results_df[results_df['is_original']].iloc[0]
        best_designed = results_df[~results_df['is_original']].iloc[0] if (~results_df['is_original']).any() else None

        if best_designed is not None:
            improvement = best_designed['ipTM'] - original_row['ipTM']

            print(f"\n📊 원본 대비 개선도:") # 한국어 유지
            print("─"*60) # 한국어 유지
            print(f"   원본 ipTM: {original_row['ipTM']:.3f}") # 한국어 유지
            print(f"   최고 설계 ipTM: {best_designed['ipTM']:.3f}") # 한국어 유지
            print(f"   개선: {improvement:+.3f} ({improvement/original_row['ipTM']*100:+.1f}%)") # 한국어 유지

            if improvement > 0:
                print("   🎯 설계가 성공적으로 결합력을 개선했습니다!") # 한국어 유지
            else:
                print("   ℹ️ 원본이 여전히 최고 성능을 보입니다.") # 한국어 유지

    # 결과 저장
    results_csv = f"{global_docking_dir}_summary.csv"
    results_df.to_csv(results_csv, index=False)
    print(f"\n💾 결과 저장: {results_csv}") # 한국어 유지

    # 전역 변수 저장
    global_docking_results = results_df

In [ ]:
#@title 🔬 3D 구조 상호작용 시각화 { display-mode: "form" }

#@markdown ### 시각화할 변이체 선택
variant_to_visualize = "all_top3" #@param ["best", "original", "all_top3"]

print("="*70)
print("🔬 항체-타겟 상호작용 3D 시각화")
print("="*70)

def visualize_docking_structure(pdb_file, variant_name, rank=1):
    """도킹 구조를 3D로 시각화"""

    print(f"\n{'🥇' if rank == 1 else '🥈' if rank == 2 else '🥉'} {variant_name}")
    print("─"*50)

    # py3Dmol 뷰어 생성
    view = py3Dmol.view(width=900, height=600)

    # PDB 파일 로드
    with open(pdb_file, 'r') as f:
        pdb_string = f.read()

    view.addModel(pdb_string, 'pdb')

    # 스타일 설정
    # 항체 Heavy Chain (A, B)
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightblue', 'opacity': 0.8}})
    view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'lightblue', 'opacity': 0.8}})

    # 항체 Light Chain (C, D)
    view.setStyle({'chain': 'C'}, {'cartoon': {'color': 'steelblue', 'opacity': 0.8}})
    view.setStyle({'chain': 'D'}, {'cartoon': {'color': 'steelblue', 'opacity': 0.8}})

    # 타겟 단백질 (E 또는 마지막 체인)
    view.setStyle({'chain': 'E'}, {'surface': {'color': 'lightgray', 'opacity': 0.6}})
    view.setStyle({'chain': 'E'}, {'cartoon': {'color': 'gray', 'opacity': 0.3}})

    # CDR 영역 강조 (IMGT numbering 기준)
    # Heavy Chain CDRs (빨간색)
    for chain in ['A', 'B']:
        view.addStyle({'chain': chain, 'resi': '27-38'},
                     {'cartoon': {'color': 'red', 'thickness': 0.6}})  # CDR-H1
        view.addStyle({'chain': chain, 'resi': '56-65'},
                     {'cartoon': {'color': 'red', 'thickness': 0.6}})  # CDR-H2
        view.addStyle({'chain': chain, 'resi': '105-117'},
                     {'cartoon': {'color': 'red', 'thickness': 0.6}})  # CDR-H3

    # Light Chain CDRs (주황색)
    for chain in ['C', 'D']:
        view.addStyle({'chain': chain, 'resi': '27-38'},
                     {'cartoon': {'color': 'orange', 'thickness': 0.6}})  # CDR-L1
        view.addStyle({'chain': chain, 'resi': '56-65'},
                     {'cartoon': {'color': 'orange', 'thickness': 0.6}})  # CDR-L2
        view.addStyle({'chain': chain, 'resi': '105-117'},
                     {'cartoon': {'color': 'orange', 'thickness': 0.6}})  # CDR-L3

    # 인터페이스 잔기 강조 (거리 기반)
    view.addStyle({'chain': 'E', 'within': {'distance': 5, 'sel': {'chain': 'A,B,C,D'}}},
                 {'stick': {'color': 'yellow', 'radius': 0.2}})

    # 라벨 추가
    view.addLabel(f"{variant_name[:20]}",
                 {'position': {'x': 0, 'y': 0, 'z': 0}},
                 {'backgroundColor': 'black', 'fontColor': 'white', 'fontSize': 14})

    # 시점 설정
    view.zoomTo()
    view.rotate(90, 'y')

    return view

# 시각화 실행
if 'global_docking_results' in globals() and not global_docking_results.empty:

    if variant_to_visualize == "best":
        # 최고 변이체만
        best = global_docking_results.iloc[0]
        view = visualize_docking_structure(best['pdb_file'], best['variant'])
        print(f"   ipTM: {best['ipTM']:.3f}")
        print(f"   pLDDT: {best['pLDDT']:.1f}")
        view.show()

    elif variant_to_visualize == "original":
        # 원본만
        if global_docking_results['is_original'].any():
            original = global_docking_results[global_docking_results['is_original']].iloc[0]
            view = visualize_docking_structure(original['pdb_file'], original['variant'] + " [원본]")
            print(f"   ipTM: {original['ipTM']:.3f}")
            print(f"   pLDDT: {original['pLDDT']:.1f}")
            view.show()
        else:
            print("⚠️ 원본 항체가 없습니다.")

    else:  # all_top3
        # 상위 3개
        for idx, row in global_docking_results.head(3).iterrows():
            rank = list(global_docking_results.index).index(idx) + 1
            view = visualize_docking_structure(row['pdb_file'], row['variant'], rank)
            print(f"   ipTM: {row['ipTM']:.3f}")
            print(f"   pLDDT: {row['pLDDT']:.1f}")
            view.show()
            print()

# 범례 표시
legend_html = """
<div style="background: white; padding: 15px; border-radius: 10px;
            border: 2px solid #e0e0e0; margin: 20px 0;">
    <h4 style="margin-top: 0; color: #333;">🎨 구조 색상 설명</h4>
    <table style="width: 100%;">
        <tr>
            <td style="padding: 5px;">
                <span style="background: lightblue; padding: 3px 10px; border-radius: 3px;">
                    ⬜ 연한 파랑
                </span>
            </td>
            <td>항체 Heavy Chain (VH)</td>
        </tr>
        <tr>
            <td style="padding: 5px;">
                <span style="background: steelblue; color: white; padding: 3px 10px; border-radius: 3px;">
                    ⬜ 진한 파랑
                </span>
            </td>
            <td>항체 Light Chain (VL)</td>
        </tr>
        <tr>
            <td style="padding: 5px;">
                <span style="background: red; color: white; padding: 3px 10px; border-radius: 3px;">
                    ⬜ 빨강
                </span>
            </td>
            <td>Heavy Chain CDR 영역</td>
        </tr>
        <tr>
            <td style="padding: 5px;">
                <span style="background: orange; padding: 3px 10px; border-radius: 3px;">
                    ⬜ 주황
                </span>
            </td>
            <td>Light Chain CDR 영역</td>
        </tr>
        <tr>
            <td style="padding: 5px;">
                <span style="background: gray; color: white; padding: 3px 10px; border-radius: 3px;">
                    ⬜ 회색
                </span>
            </td>
            <td>타겟 단백질 ({} - ChEMBL: {})</td>
        </tr>
        <tr>
            <td style="padding: 5px;">
                <span style="background: yellow; padding: 3px 10px; border-radius: 3px;">
                    ⬜ 노랑
                </span>
            </td>
            <td>인터페이스 잔기 (5Å 이내)</td>
        </tr>
    </table>
</div>
""".format(global_target_name, global_target_chembl_id)

display(HTML(legend_html))

print("\n💡 시각화 팁:")
print("   • 마우스 드래그: 회전")
print("   • 스크롤: 확대/축소")
print("   • Shift+드래그: 이동")
print("   • CDR 영역이 타겟과 접촉하는지 확인하세요")

In [ ]:
#@title 📈 최종 보고서 생성 { display-mode: "form" }

print("="*70)
print("📈 AI 기반 항체 설계 최종 보고서")
print("="*70)

# HTML 보고서 생성
report_html = f"""
<div style="background: linear-gradient(135deg, #f5f5f5 0%, #e0e0e0 100%);
            padding: 30px; border-radius: 15px; font-family: Arial, sans-serif;">

    <h1 style="color: #333; text-align: center; margin-bottom: 30px;">
        🧬 항체 설계 및 도킹 분석 보고서
    </h1>

    <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px;">
        <h2 style="color: #2196F3; border-bottom: 2px solid #2196F3; padding-bottom: 10px;">
            1. 프로젝트 개요
        </h2>
        <table style="width: 100%; margin-top: 15px;">
            <tr style="background: #f9f9f9;">
                <td style="padding: 10px; font-weight: bold; width: 30%;">타겟 단백질</td>
                <td style="padding: 10px;">{global_target_name}</td>
            </tr>
            <tr>
                <td style="padding: 10px; font-weight: bold;">ChEMBL ID</td>
                <td style="padding: 10px;">{global_target_chembl_id}</td>
            </tr>
            <tr style="background: #f9f9f9;">
                <td style="padding: 10px; font-weight: bold;">참조 PDB</td>
                <td style="padding: 10px;">{global_pdb if 'global_pdb' in globals() else 'N/A'}</td>
            </tr>
            <tr>
                <td style="padding: 10px; font-weight: bold;">선택된 항체</td>
                <td style="padding: 10px;">{selected_antibody_name if 'selected_antibody_name' in globals() else 'ProteinMPNN variants'}</td>
            </tr>
        </table>
    </div>

    <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px;">
        <h2 style="color: #4CAF50; border-bottom: 2px solid #4CAF50; padding-bottom: 10px;">
            2. 항체 설계 결과
        </h2>
        <table style="width: 100%; margin-top: 15px;">
            <tr style="background: #f9f9f9;">
                <td style="padding: 10px; font-weight: bold;">생성된 변이체 수</td>
                <td style="padding: 10px;">{num_seqs if 'num_seqs' in globals() else 'N/A'}</td>
            </tr>
            <tr>
                <td style="padding: 10px; font-weight: bold;">ConSurf 보존도 적용</td>
                <td style="padding: 10px;">{'Yes' if 'consurf_fixed_positions' in globals() else 'No'}</td>
            </tr>
            <tr style="background: #f9f9f9;">
                <td style="padding: 10px; font-weight: bold;">구조 예측 완료</td>
                <td style="padding: 10px;">{len(predicted_pdbs) if 'predicted_pdbs' in locals() else 0}개</td>
            </tr>
        </table>
    </div>
"""

# 도킹 결과 추가
if 'global_docking_results' in globals() and not global_docking_results.empty:
    best = global_docking_results.iloc[0]

    # 품질 평가
    if best['ipTM'] > 0.8:
        quality_badge = '<span style="background: #4CAF50; color: white; padding: 5px 10px; border-radius: 5px;">Excellent</span>'
        recommendation = "실험적 검증을 위한 최우선 후보"
    elif best['ipTM'] > 0.6:
        quality_badge = '<span style="background: #FFC107; color: white; padding: 5px 10px; border-radius: 5px;">Good</span>'
        recommendation = "추가 최적화 후 실험 검증 고려"
    else:
        quality_badge = '<span style="background: #FF9800; color: white; padding: 5px 10px; border-radius: 5px;">Moderate</span>'
        recommendation = "추가 설계 개선 필요"

    report_html += f"""
    <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px;">
        <h2 style="color: #9C27B0; border-bottom: 2px solid #9C27B0; padding-bottom: 10px;">
            3. 도킹 분석 결과
        </h2>
        <table style="width: 100%; margin-top: 15px;">
            <tr style="background: #f9f9f9;">
                <td style="padding: 10px; font-weight: bold;">최고 변이체</td>
                <td style="padding: 10px;">{best['variant'][:30]}</td>
            </tr>
            <tr>
                <td style="padding: 10px; font-weight: bold;">ipTM 점수</td>
                <td style="padding: 10px;">{best['ipTM']:.3f}</td>
            </tr>
            <tr style="background: #f9f9f9;">
                <td style="padding: 10px; font-weight: bold;">품질 평가</td>
                <td style="padding: 10px;">{quality_badge}</td>
            </tr>
            <tr>
                <td style="padding: 10px; font-weight: bold;">권장사항</td>
                <td style="padding: 10px;">{recommendation}</td>
            </tr>
        </table>
    </div>
    """

    # 상위 5개 변이체 테이블
    report_html += """
    <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px;">
        <h2 style="color: #FF5722; border-bottom: 2px solid #FF5722; padding-bottom: 10px;">
            4. 상위 변이체 순위
        </h2>
        <table style="width: 100%; margin-top: 15px; border-collapse: collapse;">
            <thead>
                <tr style="background: #f0f0f0;">
                    <th style="padding: 10px; text-align: left; border: 1px solid #ddd;">순위</th>
                    <th style="padding: 10px; text-align: left; border: 1px solid #ddd;">변이체</th>
                    <th style="padding: 10px; text-align: center; border: 1px solid #ddd;">ipTM</th>
                    <th style="padding: 10px; text-align: center; border: 1px solid #ddd;">pLDDT</th>
                </tr>
            </thead>
            <tbody>
    """

    for idx, row in global_docking_results.head(5).iterrows():
        rank = list(global_docking_results.index).index(idx) + 1
        emoji = "🥇" if rank == 1 else ("🥈" if rank == 2 else ("🥉" if rank == 3 else f"{rank}"))
        original_tag = " 📌" if row['is_original'] else ""

        report_html += f"""
                <tr style="{'background: #fffde7;' if row['is_original'] else ''}">
                    <td style="padding: 10px; border: 1px solid #ddd;">{emoji}</td>
                    <td style="padding: 10px; border: 1px solid #ddd;">{row['variant'][:25]}{original_tag}</td>
                    <td style="padding: 10px; text-align: center; border: 1px solid #ddd;">{row['ipTM']:.3f}</td>
                    <td style="padding: 10px; text-align: center; border: 1px solid #ddd;">{row['pLDDT']:.1f}</td>
                </tr>
        """

    report_html += """
            </tbody>
        </table>
    </div>
    """

# 보고서 마무리
report_html += """
    <div style="background: #e8f5e9; padding: 20px; border-radius: 10px; margin-top: 30px;">
        <h3 style="color: #2e7d32; margin-top: 0;">✅ 프로젝트 완료</h3>
        <p style="color: #333; line-height: 1.6;">
            AI 기반 항체 설계 파이프라인이 성공적으로 완료되었습니다.<br>
            생성된 변이체들은 타겟 단백질과의 도킹 시뮬레이션을 통해 평가되었습니다.<br>
            상위 순위 변이체들을 실험적 검증 후보로 고려하시기 바랍니다.
        </p>
    </div>
</div>
"""

# 보고서 표시
display(HTML(report_html))

# 파일 다운로드 준비
print("\n📥 결과 파일 다운로드")
print("─"*60)

# 모든 결과를 ZIP으로 압축
import zipfile
from google.colab import files

zip_filename = f"antibody_design_{global_target_name.replace(' ', '_')}_results.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # 도킹 결과 CSV
    if 'global_docking_results' in globals():
        results_csv = f"docking_results_{global_target_name.replace(' ', '_')}.csv"
        global_docking_results.to_csv(results_csv, index=False)
        zipf.write(results_csv)

    # HTML 보고서
    report_file = f"report_{global_target_name.replace(' ', '_')}.html"
    with open(report_file, 'w') as f:
        f.write(report_html)
    zipf.write(report_file)

    # 최고 PDB 구조
    if 'global_docking_results' in globals() and not global_docking_results.empty:
        best_pdb = global_docking_results.iloc[0]['pdb_file']
        if os.path.exists(best_pdb):
            zipf.write(best_pdb, arcname=f"best_docking_{os.path.basename(best_pdb)}")

print(f"✅ 결과 압축 완료: {zip_filename}")
files.download(zip_filename)

print("\n🎉 모든 작업이 완료되었습니다!")
print("   다운로드된 파일을 확인하세요.")